# Reproducing Henry & Kohn (2022) from CRCNS v4-2

**Paper.** Henry CA, Kohn A (2022). Feature representation under crowding in macaque V1 and V4
neuronal populations. Current Biology 32, 5126-5137. doi:10.1016/j.cub.2022.10.049. The claims
file `claims/henry-2022.yaml` was written from the version of record (main text, STAR Methods
and Supplemental Information, as extracted from the PDF).

**Dataset.** CRCNS v4-2, "Simultaneous extracellular recordings from visual areas V1 and V4 from
macaque monkeys viewing crowded displays" (doi:10.6080/K0736P4H), the paper's deposit. Files are
streamed from the public S3 bucket `crcnsarchive` under `v4-2/live/`; the zip is opened in place
and nothing is written to disk.

**Version.** CRCNS data sets are not versioned (docs/archives.md), and the paper cites the data
set by its DOI only, so the version pinned is the one published under `v4-2/live/` (the bucket
holds no other), recorded as `unversioned`. Its data description is version 0.7 (9 Nov 2022). The
MD5 sums from the data set's `checksums.md5` for the files read are: `data.zip`
bdd60f999b997523151b9ae24a42b837 (checked against the bytes read, below) and
`docs/crcns_v4-2_data_description.pdf` 40f2f30456a87ad3999e8a929a424220.

**What the data set holds.** Spike counts (0 to 0.25 s after each stimulus onset) of the
populations analysed in the paper: the separation experiment (11 sessions, 22 populations per
area, 396 V1 and 691 V4 units) and the distractor-orientation experiment (4 sessions, Figure 6E).
It holds no receptive-field mapping sessions, no per-session coarse mapping of targets and single
distractors, and no waveforms.

The pseudo-population analysis (figs3-pseudo-populations) is in the second notebook,
`pseudo_populations.ipynb`, to keep each notebook within the run time limit.

| Claim | Tested here |
|---|---|
| data-populations | yes |
| fig1b-v4-rf-larger | blocked, data_missing (no mapping sessions) |
| fig2a-v1-single-neuron-near, fig2b-v4-single-neuron-near | yes |
| fig2c-farthest-spacing-slopes, fig2c-v4-worse-all-spacings | yes |
| fig2de-population-decoding-example | yes |
| fig2f-v1-threshold-elevation, fig2f-v4-threshold-elevation | yes |
| figs3-pseudo-populations | yes, in `pseudo_populations.ipynb` |
| fig3ab-correlations-reduced | yes |
| fig3c-shuffled-threshold-elevation | yes |
| fig3d-v1-fano-closest, fig3e-v4-fano-all-spacings | yes |
| fig4a-v1-modulation-indices, fig4b-v4-modulation-indices | yes |
| fig4c-modulation-strength-significant, fig4c-v4-stronger-than-v1, fig4c-v4-maintained-v1-weaker-far | yes |
| figs4-rf-alignment | blocked, data_missing (no coarse mapping of each distractor alone) |
| figs5-additive-vs-multiplicative | yes |
| fig5bc-tuning-model-terms, fit-quality, fig5-terms-covary, fig5-negative-multiplicative | yes |
| fig5de-simulated-discriminability | yes |
| fig6c-tuning-correlation-near, fig6d-tuning-correlation-spacing, fig6e-tuning-correlation-configuration | yes |
| fig6-well-isolated | blocked, data_missing (no waveform SNR) |
| fig7cd-fixed-decoder-generalization | yes |
| figs6-common-neuron-pool | yes |

Method details the claims file leaves to the paper are cited in the helper `crowding.py` and in
each section. All analyses use every deposited unit: the deposit holds exactly the paper's 396
and 691 units, and the selection rule of the Methods does not regenerate them (data-populations).

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
import resource
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats

import crowding as cw

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
T0 = time.time()
print(cw.DATASET)
D = cw.load()
print("MD5 of data.zip as read:", D["md5"], "; checksums.md5 gives", cw.DATA_MD5,
      "; match:", D["md5"] == cw.DATA_MD5)
S, O = D["separation"], D["orientation"]
SP = cw.SPACINGS
SPL = {1.32: "1.3", 1.64: "1.6", 2.04: "2.1", 2.56: "2.6"}
AREAS = ("V1", "V4")


def pops(area, exp=None):
    return [p for p in (exp or S) if p["area"] == area]


def ttest_1s_greater(a, b, equal_var=True):
    return stats.ttest_ind(a, b, equal_var=equal_var, alternative="greater")

## Data check

Each check of the description against the values is stated under its output.

In [ ]:
rows = []
for exp, ps in (("separation", S), ("orientation", O)):
    for p in ps:
        k = p["p"].kind.value_counts()
        rows.append(dict(experiment=exp, area=p["area"], animal=p["animal"], session=p["session"],
                         orange=p["orange"], neurons=p["counts"].shape[0], epochs=p["counts"].shape[1],
                         blank=k.get("blank", 0), dist_only=k.get("dist", 0), alone=k.get("alone", 0),
                         crowd=k.get("crowd", 0), reference=p["ref"],
                         target_rel=" ".join(f"{x:g}" for x in p["oris"] - p["ref"]),
                         count_min=p["counts"].min(), count_max=p["counts"].max()))
info = pd.DataFrame(rows)
print(info.to_string())
print()
print("Populations, sessions and neurons by experiment, area and animal:")
print(info.groupby(["experiment", "area", "animal"]).agg(populations=("session", "size"),
      sessions=("session", "nunique"), neurons=("neurons", "sum"), epochs=("epochs", "sum")).to_string())
print()
for exp, ps in (("separation", S), ("orientation", O)):
    allp = pd.concat([p["p"] for p in ps])
    print(exp, "spacings:", sorted(allp.sep.dropna().unique()), "; target contrast:",
          sorted(allp.target_contrast.unique()), "; distractor contrast:", sorted(allp.distractor_contrast.unique()))
    combos = set()
    for p in ps:
        d = p["p"][p["p"].kind == "crowd"]
        combos |= set(map(tuple, (d[[f"distractor{i}_ori" for i in range(1, 5)]].values - p["ref"]).astype(int)))
    print("   distractor orientations relative to the reference (d1, d2, d3, d4):", sorted(combos))
print()
per_cond = pd.concat([p["p"].assign(area=p["area"], pop=i % 22).query("kind in ['alone','crowd']")
                      .groupby(["area", "pop", "kind", "sep", "oi"], dropna=False).size()
                      for i, p in enumerate(S)])
print("Separation experiment, epochs per stimulus condition: min", per_cond.min(), "median",
      per_cond.median(), "max", per_cond.max())
print("Spike counts: integers from", min(p["counts"].min() for p in S + O), "to",
      max(p["counts"].max() for p in S + O), "spikes per 0.25 s epoch")

The description says 15 sessions in two files, populations per area and orientation range, spike
counts per stimulus epoch (neurons x epochs) and fields for session and animal; the values agree:
11 sessions (7 from AL, 4 from CA) and 22 populations per area in the separation file, 4 sessions
(2 per animal) and 8 populations per area in the orientation file. The paper's design is there:
four target orientations at -9, -3, +3 and +9 degrees from the reference, isolated or with four
distractors at +/-10 degrees alternating around the target, at four spacings (stored as 1.32,
1.64, 2.04 and 2.56 degrees, printed in the paper as 1.3, 1.6, 2.1 and 2.6), and two references
90 degrees apart per session. The files also hold blank epochs and distractor-only epochs, which
the description does not mention; blank epochs are taken as the spontaneous condition. The
orientation experiment has seven target orientations (-12 to +12 degrees in steps of 4), alone or
with one of two distractor configurations whose orientations are swapped (+10/-10 against
-10/+10), at one spacing that the file does not state.

In [ ]:
# Are the V1 and V4 epochs of a population the same epochs in the same order?
rows = []
for a, b in zip(pops("V1"), pops("V4")):
    ka = a["p"].fillna(-1).groupby(["kind", "oi", "sep"]).size()
    kb = b["p"].fillna(-1).groupby(["kind", "oi", "sep"]).size()
    same_seq = (a["p"].fillna(-1)[["kind", "oi", "sep"]].values == b["p"].fillna(-1)[["kind", "oi", "sep"]].values).all(1)
    rows.append(dict(session=a["session"], orange=a["orange"], epochs=len(a["p"]),
                     same_counts_per_condition=ka.equals(kb), same_condition_at_same_index=same_seq.mean()))
al = pd.DataFrame(rows)
print(al.to_string())
print("Expected share of equal conditions at the same index if orders were unrelated: about",
      round(float(np.mean([(p["p"].fillna(-1).groupby(["kind", "oi", "sep"]).size() / len(p["p"])).pow(2).sum()
                           for p in pops("V1")])), 3))


def resid_series(p):
    """Population mean count per epoch minus the mean of its condition."""
    c = p["counts"].astype(float)
    key = p["p"].kind + p["p"].oi.astype(str) + p["p"].sep.fillna(-1).astype(str)
    r = c.copy()
    for k in key.unique():
        m = (key == k).values
        r[:, m] = c[:, m] - c[:, m].mean(1, keepdims=True)
    return r.mean(0)


def lagcorr(x, lag):
    return np.corrcoef(x[:-lag], x[lag:])[0, 1]


rows = []
for p in S:
    x = resid_series(p)
    rows.append(dict(area=p["area"], session=p["session"], orange=p["orange"],
                     lag1=lagcorr(x, 1), lag10=lagcorr(x, 10), lag50=lagcorr(x, 50),
                     pair_even=np.corrcoef(x[0:-1:2], x[1::2])[0, 1],
                     pair_odd=np.corrcoef(x[1:-1:2], x[2::2])[0, 1]))
ac = pd.DataFrame(rows)
print()
print("Autocorrelation of the population residual over the stored epoch order (median over populations):")
print(ac.groupby("area")[["lag1", "lag10", "lag50", "pair_even", "pair_odd"]].median().round(3).to_string())
print("Populations with lag-10 autocorrelation above 0.1: V1", int((ac.query("area=='V1'").lag10 > 0.1).sum()),
      "of 22, V4", int((ac.query("area=='V4'").lag10 > 0.1).sum()), "of 22")

Within each session and orientation range the V1 and V4 entries have the same number of epochs of
every condition, but not in the same order: the share of epochs whose condition agrees at the same
index is at the level expected for unrelated orders. The V1 epochs carry slow fluctuations of the
population response (residual autocorrelation at a lag of 10 epochs), as recordings in time order
do; the V4 epochs carry none. So the V1 epochs are consistent with the recording order and the V4
epochs are stored in some other order, and epochs cannot be matched between the areas. No claim
tested here pairs V1 and V4 epochs. The two epochs of one trial cannot be told apart: in V1 the
residuals of epochs 2k and 2k+1 are no more alike than those of 2k+1 and 2k+2. The description
says nothing about either point.

In [ ]:
# Clock check: the counts must follow the stimulus labels. Contrast target-alone with blank epochs,
# with the true labels and with the labels shifted by one and by five epochs.
rng = np.random.default_rng(1)
rows = []
for p in S:
    kind = p["p"].kind.values
    for shift in (0, 1, 5):
        k = np.roll(kind, shift)
        d = p["counts"][:, k == "alone"].mean(1) - p["counts"][:, k == "blank"].mean(1)
        rows.append(dict(area=p["area"], shift=shift, mean_diff=d.mean(), frac_pos=(d > 0).mean()))
ck = pd.DataFrame(rows).groupby(["area", "shift"]).agg(mean_alone_minus_blank=("mean_diff", "mean"),
                                                      frac_neurons_alone_above_blank=("frac_pos", "mean"))
print(ck.round(3).to_string())
# Orientation labels: single-neuron AUC between the extreme orientations, true against shifted labels
rows = []
for p in S:
    oi = p["p"].oi.values.copy()
    kind = p["p"].kind.values
    for shift in (0, 1):
        o = np.roll(oi, shift)
        a = p["counts"][:, (kind == "alone") & (o == 0)]
        b = p["counts"][:, (kind == "alone") & (o == 3)]
        au = np.array([cw.auc(a[i], b[i]) for i in range(a.shape[0])])
        rows.append(dict(area=p["area"], shift=shift, mean_abs_auc_dev=np.mean(np.abs(au - 0.5))))
print(pd.DataFrame(rows).groupby(["area", "shift"]).mean_abs_auc_dev.mean().round(3).to_string())

Responses follow the labels: with the true labels target-alone counts exceed blank counts, and
the extreme orientations are told apart more than with labels shifted by one epoch, in both areas.
With the labels shifted the target and blank difference vanishes. The labels and counts share one
order within each area. The next cell breaks this down by population.

In [ ]:
# Blank and distractor-only epochs against target epochs, per population: mean over neurons of
# (mean count - blank mean) / blank SD, and the orientation tuning of target-alone counts (mean
# over neurons of the variance of the four orientation means over the mean within-orientation
# variance of a mean; about 1 without tuning), with true labels and labels shifted by one epoch.
rows = []
for p in S:
    c, P = p["counts"], p["p"]
    bl = c[:, cw.stim_mask(p, "blank")]
    sd = bl.std(1, ddof=1)
    sd[sd == 0] = 1
    r = dict(area=p["area"], animal=p["animal"], session=p["session"], orange=p["orange"], blank_mean=bl.mean())
    for kind, s, lab in (("alone", None, "z_alone"), ("dist", None, "z_dist_only"), ("crowd", 1.32, "z_crowd_1.3")):
        r[lab] = np.mean((c[:, cw.stim_mask(p, kind, s)].mean(1) - bl.mean(1)) / sd)
    for shift in (0, 1):
        kind = np.roll(P.kind.values, shift)
        oi = np.roll(P.oi.values, shift)
        ms = [(kind == "alone") & (oi == o) for o in range(4)]
        means = np.stack([c[:, m].mean(1) for m in ms], 1)
        within = np.mean([c[:, m].var(1) / m.sum() for m in ms], 0)
        r[f"tuning_F_shift{shift}"] = np.nanmean(means.var(1, ddof=1) / within)
    rows.append(r)
BL = pd.DataFrame(rows)
print(BL.round(2).to_string())
flat = BL.groupby("session").z_alone.max() < 0.5
print("Sessions whose target epochs do not exceed the blank epochs (z < 0.5 in every population):",
      list(flat[flat].index))

In [ ]:
# Per population: the contrast values the parameters hold, and the mean count of the blank,
# distractor-only and target-alone epochs, with the fraction of units whose target-alone (and
# distractor-only) mean exceeds their blank mean.
rows = []
for p in S:
    P, c = p["p"], p["counts"]
    m = {k: c[:, cw.stim_mask(p, k)].mean(1) for k in ("blank", "dist", "alone")}
    rows.append(dict(area=p["area"], session=p["session"], orange=p["orange"],
                     target_contrasts=" ".join(f"{x:g}" for x in np.sort(P.target_contrast.unique())),
                     distractor_contrasts=" ".join(f"{x:g}" for x in np.sort(P.distractor_contrast.unique())),
                     n_blank=int((P.kind == "blank").sum()), n_dist=int((P.kind == "dist").sum()),
                     mean_blank=m["blank"].mean(), mean_dist=m["dist"].mean(), mean_alone=m["alone"].mean(),
                     frac_alone_gt_blank=np.mean(m["alone"] > m["blank"]),
                     frac_dist_gt_blank=np.mean(m["dist"] > m["blank"])))
CT = pd.DataFrame(rows)
print(CT.round(3).to_string())
flat_s = [1, 2, 8, 9, 10, 11]
for area in AREAS:
    d = CT[CT.area == area]
    for lab, sel in (("sessions 1, 2, 8-11", d.session.isin(flat_s)), ("sessions 3-7", ~d.session.isin(flat_s))):
        x = d[sel]
        print(f"{area} {lab}: fraction of units with target-alone mean above blank {x.frac_alone_gt_blank.min():.2f} "
              f"to {x.frac_alone_gt_blank.max():.2f}; distractor-only above blank {x.frac_dist_gt_blank.min():.2f} to "
              f"{x.frac_dist_gt_blank.max():.2f}")


In sessions 1, 2 and 8 to 11 (12 of the 22 populations in each area, both animals) the epochs
labelled blank (target and distractor contrast 0) and those labelled distractor-only give the
same counts as the target epochs (z near 0), while in sessions 3 to 7 the targets drive the
neurons well above the blank epochs. The labels are aligned with the counts in all sessions:
target-alone counts depend on the target orientation with the true labels (tuning F above 1)
and not with labels shifted by one epoch. The contrast parameters hold the same two values (0 and
0.5) for targets and distractors in every session, so nothing in the labels marks the six
sessions as different; only the counts do. In sessions 3 to 7 every unit's target-alone mean
exceeds its blank mean; in the other six sessions 8% to 86% of V1 units and 23% to 75% of V4
units do, around the half expected when the two kinds of epoch do not differ. So in six sessions
the epochs labelled blank cannot have been blank, or their counts are not those of the epochs the
labels describe; the files cannot tell which. Spontaneous activity cannot be measured there. This
is recorded as a finding; it bears on the selection rule (data-populations) and on the driven
fractions of figs6-common-neuron-pool.


## data-populations

> Eleven sessions gave 22 neuronal populations in each area, of on average 19 +/- 1.9 V1 neurons
> and 31 +/- 3.4 V4 neurons, 396 V1 and 691 V4 neurons in the single-neuron analyses.

The selection rule ("responses >= 1 s.d. above spontaneous rate", STAR Methods) is applied to the
deposited units under the two readings of the criterion, with blank epochs as the spontaneous
condition (the only spontaneous data held) and the SD over blank epochs.

In [ ]:
rows = []
for area in AREAS:
    ps = pops(area)
    sizes = np.array([p["counts"].shape[0] for p in ps])
    sel_a = sel_b = 0
    sizes_a, sizes_b = [], []
    for p in ps:
        bl = p["counts"][:, cw.stim_mask(p, "blank")]
        thr = bl.mean(1) + bl.std(1, ddof=1)
        a = p["counts"][:, cw.stim_mask(p, "alone")].mean(1) >= thr
        b = cw.tuning(p, "alone").max(1) >= thr
        sizes_a.append(a.sum())
        sizes_b.append(b.sum())
    for label, sz in (("deposited units", sizes), ("selection (a), mean over targets", np.array(sizes_a)),
                      ("selection (b), best target", np.array(sizes_b))):
        rows.append(dict(area=area, units=label, sessions=len({p["session"] for p in ps}), populations=len(ps),
                         mean_size=sz.mean(), sem_size=sz.std(ddof=1) / np.sqrt(len(sz)), total=sz.sum()))
dp = pd.DataFrame(rows)
print(dp.round(2).to_string())
print()
print("Deposited units per animal (Figure S7: V1 264 and 132, V4 288 and 403):")
print(info.query("experiment=='separation'").groupby(["area", "animal"]).neurons.sum().to_string())
print()
print("Expected: 11 sessions, 22 populations, V1 19 +/- 1.9, V4 31 +/- 3.4, totals 396 +/- 40 and 691 +/- 69")
for _, r in dp.iterrows():
    exp_size, tol_size, exp_tot, tol_tot = (19, 1.9, 396, 40) if r.area == "V1" else (31, 3.4, 691, 69)
    print(f"{r.area} {r.units:34s} size within: {abs(r.mean_size - exp_size) <= tol_size}; "
          f"total within: {abs(r.total - exp_tot) <= tol_tot}")

**Result: partially reproduced.** The data set holds 11 sessions and 22 populations per area, and
its units number exactly 396 in V1 and 691 in V4 (264 and 132, 288 and 403 per animal, as in
Figure S7), with mean population sizes of 18.0 +/- 1.7 and 31.4 +/- 3.4, within tolerance. But
the deposited units are the selected ones, and the Methods' rule, applied to them under either
reading of the criterion, keeps only 115 or 133 of the V1 units and 147 or 174 of the V4 units,
far from the reported counts. One reason is in the data check: in sessions 1, 2 and 8 to 11 the
epochs labelled blank respond as the target epochs do, so no unit there passes. The counts hold
for the data as deposited; the stated selection does not regenerate them.

## Single-neuron discriminability (Figure 2A to 2C)

> fig2a-v1-single-neuron-near: V1 single-neuron discriminability of target orientation is lower
> with the nearest distractors than for targets alone, with a regression slope of 0.60 (95% CI
> 0.50 to 0.69).
>
> fig2b-v4-single-neuron-near: In V4 the loss of single-neuron discriminability with the nearest
> distractors is stronger, with a regression slope of 0.20 (95% CI 0.14 to 0.26).
>
> fig2c-farthest-spacing-slopes: At the farthest spacing ... slopes of 0.74 (0.66 to 0.82) in V1
> and 0.44 (0.37 to 0.50) in V4.
>
> fig2c-v4-worse-all-spacings: Distractors impair single-neuron discriminability more in V4 than
> in V1 at every target-distractor spacing.

All deposited units are used (n = 396 and 691, as in Figure 2). AUC of the two extreme target
orientations, the one closer to 0 degrees taken as the preferred (Results, paragraph on Figure
2A); OLS slope of crowded on target-alone AUC; 95% CI from 1000 bootstrap samples over neurons. "Closer
to 0" is read as the axial orientation; in the paper's frame of orientations relative to the
reference (-9 to +9 degrees) the two extremes are equally close to 0, so the reading with the
lower orientation always taken as preferred is shown as a variant (`slope_lower_orientation_first`).

In [ ]:
rng = np.random.default_rng(2)
AUC = {}
for area in AREAS:
    for order in ("paper", "lower_first"):
        AUC[(area, order, None)] = np.concatenate([cw.neuron_aucs(p, "alone", order=order) for p in pops(area)])
        for s in SP:
            AUC[(area, order, s)] = np.concatenate([cw.neuron_aucs(p, "crowd", s, order=order) for p in pops(area)])
animal = {area: np.concatenate([[p["animal"]] * p["counts"].shape[0] for p in pops(area)]) for area in AREAS}
rows = []
for area in AREAS:
    for s in SP:
        x, y = AUC[(area, "paper", None)], AUC[(area, "paper", s)]
        sl, lo, hi = cw.slope_ci(x, y, rng)
        so, slo, shi = cw.slope_ci(x, y, rng, fit=cw.origin_slope)
        sv = cw.ols_slope(AUC[(area, "lower_first", None)], AUC[(area, "lower_first", s)])
        r = dict(area=area, spacing=SPL[s], n=len(x), slope=sl, ci_low=lo, ci_high=hi,
                 slope_through_half=so, ci_low_through_half=slo, ci_high_through_half=shi,
                 slope_lower_orientation_first=sv)
        for an in ("AL", "CA"):
            m = animal[area] == an
            r[f"slope_{an}"] = cw.ols_slope(x[m], y[m])
        rows.append(r)
f2 = pd.DataFrame(rows)
print(f2.round(3).to_string())
print()
print("AUC ranges: alone V1", np.round([AUC[('V1', 'paper', None)].min(), AUC[('V1', 'paper', None)].max()], 3),
      "V4", np.round([AUC[('V4', 'paper', None)].min(), AUC[('V4', 'paper', None)].max()], 3))
exp = {("V1", 1.32): (0.60, 0.10), ("V4", 1.32): (0.20, 0.06), ("V1", 2.56): (0.74, 0.08), ("V4", 2.56): (0.44, 0.07)}
for (area, s), (v, tol) in exp.items():
    r = f2[(f2.area == area) & (f2.spacing == SPL[s])].iloc[0]
    print(f"{area} {SPL[s]} deg: slope {r.slope:.3f} [{r.ci_low:.3f}, {r.ci_high:.3f}] against {v} +/- {tol}: "
          f"within {abs(r.slope - v) <= tol}; CI upper bound below 1: {r.ci_high < 1}")
print()
for s in SP:
    a = f2[(f2.area == "V1") & (f2.spacing == SPL[s])].iloc[0]
    b = f2[(f2.area == "V4") & (f2.spacing == SPL[s])].iloc[0]
    print(f"{SPL[s]} deg: (a) V4 slope below V1: {b.slope < a.slope}; (b) CIs do not overlap: {b.ci_high < a.ci_low}")

fig, axs = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, area in zip(axs, AREAS):
    ax.scatter(AUC[(area, "paper", None)], AUC[(area, "paper", 1.32)], s=6, alpha=0.5)
    ax.plot([0, 1], [0, 1], "k:")
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="AUC target alone", ylabel="AUC nearest distractors", title=area)
for area, c in zip(AREAS, ("tab:blue", "tab:red")):
    d = f2[f2.area == area]
    axs[2].errorbar(np.arange(4), d.slope, yerr=[d.slope - d.ci_low, d.ci_high - d.slope], color=c, label=area, marker="o")
axs[2].set(xticks=range(4), xticklabels=[SPL[s] for s in SP], xlabel="spacing (deg)", ylabel="regression slope", ylim=(0, 1))
axs[2].legend()
plt.tight_layout()
plt.show()
print("All AUC values lie in [0, 1], inside the axes.")

**Result: fig2a-v1-single-neuron-near reproduced.** The V1 slope at 1.3 degrees is 0.580 (95% CI
0.486 to 0.669), within 0.60 +/- 0.10, with the CI below 1; 0.607 through (0.5, 0.5). With the lower
orientation always preferred the slopes are 0.602 and 0.200 at 1.3 degrees and 0.742 and 0.440 at
2.6 (V1 and V4), within 0.022 of the main reading, so no outcome below depends on it.

**Result: fig2b-v4-single-neuron-near reproduced.** The V4 slope at 1.3 degrees is 0.206 (0.145 to
0.269), within 0.20 +/- 0.06.

**Result: fig2c-farthest-spacing-slopes reproduced.** At 2.6 degrees the slopes are 0.728 (0.647 to
0.798) in V1 and 0.437 (0.368 to 0.501) in V4, within 0.74 +/- 0.08 and 0.44 +/- 0.07.

**Result: fig2c-v4-worse-all-spacings reproduced.** At every spacing the V4 slope is below the V1
slope and the two 95% CIs do not overlap (reading b). Per animal the V4 slope is below the V1
slope at every spacing in both animals (AL 0.319 to 0.514 against 0.490 to 0.582; CA 0.122 to
0.379 against 0.644 to 0.874).

## fig3ab-correlations-reduced

> Distractors reduce noise correlations, from 0.07 +/- 0.003 to 0.05 +/- 0.002 in V1 and from
> 0.11 +/- 0.001 to 0.06 +/- 0.001 in V4 with the nearest distractors (paired t test, p < 0.001).

Pairs within each population; per stimulus, trials with both counts within 3 SD of their means;
averaged over the four target orientations (STAR Methods). Reading (a), every stored epoch as a
presentation, is the only one that can be run: the files do not say which epoch of a trial is the
first, and the residuals do not reveal the pairing (data check), so reading (b) cannot be formed.

In [ ]:
RSC = {}
for area in AREAS:
    RSC[(area, None)] = np.concatenate([cw.pair_rsc(p, "alone") for p in pops(area)])
    for s in SP:
        RSC[(area, s)] = np.concatenate([cw.pair_rsc(p, "crowd", s) for p in pops(area)])
rows = []
for area in AREAS:
    a0 = RSC[(area, None)]
    for s in [None] + list(SP):
        x = RSC[(area, s)]
        ok = np.isfinite(a0) & np.isfinite(x)
        t = stats.ttest_rel(a0[ok], x[ok]) if s is not None else None
        rows.append(dict(area=area, condition="target alone" if s is None else SPL[s], pairs=len(x),
                         pairs_finite=int(np.isfinite(x).sum()), mean=np.nanmean(x),
                         sem=np.nanstd(x, ddof=1) / np.sqrt(np.isfinite(x).sum()),
                         paired_t=None if t is None else t.statistic, p=None if t is None else t.pvalue,
                         n_paired=int(ok.sum())))
f3 = pd.DataFrame(rows)
print(f3.to_string(float_format=lambda v: f"{v:.4g}"))
print()
for area, (e0, e1) in (("V1", (0.07, 0.05)), ("V4", (0.11, 0.06))):
    r0 = f3[(f3.area == area) & (f3.condition == "target alone")].iloc[0]
    r1 = f3[(f3.area == area) & (f3.condition == "1.3")].iloc[0]
    print(f"{area}: alone {r0['mean']:.4f} (expected {e0} +/- 0.005, within {abs(r0['mean'] - e0) <= 0.005}); "
          f"nearest {r1['mean']:.4f} (expected {e1} +/- 0.005, within {abs(r1['mean'] - e1) <= 0.005}); "
          f"decrease p = {r1.p:.2g}; pairs {r0.pairs} (expected {4028 if area == 'V1' else 13020} +/- 10%)")
for area in AREAS:
    print(area, "rSC per animal, alone and nearest:",
          {an: (round(float(np.nanmean(np.concatenate([cw.pair_rsc(p, 'alone') for p in pops(area) if p['animal'] == an]))), 4),
                round(float(np.nanmean(np.concatenate([cw.pair_rsc(p, 'crowd', 1.32) for p in pops(area) if p['animal'] == an]))), 4))
           for an in ("AL", "CA")})

**Result: reproduced.** Mean rSC falls from 0.0672 +/- 0.0025 to 0.0479 +/- 0.0023 in V1 and
from 0.1122 +/- 0.0013 to 0.0555 +/- 0.0010 in V4 with the nearest distractors, all four within
0.005 of the paper's values; paired t tests give p = 2e-21 (V1) and p below 1e-300 (V4, printed
0). There are 4036 V1 and 13101 V4 pairs against 4028 and 13020. Only reading (a), every epoch
as a presentation, could be run. The decrease is also present in each animal.

## Fano factors (Figure 3D and 3E)

> fig3d-v1-fano-closest: In V1, distractors at the closest spacing slightly reduce variability,
> the Fano factor falling from 1.05 +/- 0.02 for targets alone to 0.98 +/- 0.01 (paired t test,
> p < 0.001).
>
> fig3e-v4-fano-all-spacings: In V4, distractors at all spacings significantly reduce variability,
> the Fano factor falling from 1.00 +/- 0.01 for targets alone to 0.93 +/- 0.01 with the closest
> distractors (paired t test, p < 0.001 for each spacing).

Per neuron, the geometric mean over the four stimulus conditions of variance over mean; across
neurons the geometric mean, with the SEM of the logs transformed back (exp(mean +/- SEM) given as
a half-width). Conditions with zero mean or zero variance are left out (a standard choice); the
paired t test is run on the Fano factors and on their logs.

In [ ]:
FF = {}
rows = []
for area in AREAS:
    FF[(area, None)] = np.concatenate([cw.fano(p, "alone") for p in pops(area)])
    for s in SP:
        FF[(area, s)] = np.concatenate([cw.fano(p, "crowd", s) for p in pops(area)])
    a0 = FF[(area, None)]
    for s in [None] + list(SP):
        x = FF[(area, s)]
        lx = np.log(x[np.isfinite(x)])
        gm, sem = np.exp(lx.mean()), lx.std(ddof=1) / np.sqrt(len(lx))
        r = dict(area=area, condition="target alone" if s is None else SPL[s], n=int(np.isfinite(x).sum()),
                 geo_mean=gm, sem_halfwidth=gm * (np.exp(sem) - np.exp(-sem)) / 2)
        if s is not None:
            ok = np.isfinite(a0) & np.isfinite(x)
            t1 = stats.ttest_rel(a0[ok], x[ok])
            t2 = stats.ttest_rel(np.log(a0[ok]), np.log(x[ok]))
            r.update(t_ff=t1.statistic, p_ff=t1.pvalue, t_log=t2.statistic, p_log=t2.pvalue)
        rows.append(r)
ff = pd.DataFrame(rows)
print(ff.to_string(float_format=lambda v: f"{v:.4g}"))
print("Neurons left out (no usable condition): ", {k: int((~np.isfinite(v)).sum()) for k, v in FF.items() if k[1] in (None, 1.32)})
print("Fano factor range over neurons, V1 alone", np.round([np.nanmin(FF[('V1', None)]), np.nanmax(FF[('V1', None)])], 3),
      "V4 alone", np.round([np.nanmin(FF[('V4', None)]), np.nanmax(FF[('V4', None)])], 3))
print()
for area, e0, e1, tol0, tol1 in (("V1", 1.05, 0.98, 0.02, 0.01), ("V4", 1.00, 0.93, 0.01, 0.01)):
    r0 = ff[(ff.area == area) & (ff.condition == "target alone")].iloc[0]
    r1 = ff[(ff.area == area) & (ff.condition == "1.3")].iloc[0]
    print(f"{area}: alone {r0.geo_mean:.3f} (expected {e0} +/- {tol0}: {abs(r0.geo_mean - e0) <= tol0}); "
          f"closest {r1.geo_mean:.3f} (expected {e1} +/- {tol1}: {abs(r1.geo_mean - e1) <= tol1})")
for area in AREAS:
    for an in ("AL", "CA"):
        m = animal[area] == an
        print(area, an, "geometric mean alone %.3f, closest %.3f" % (np.exp(np.nanmean(np.log(FF[(area, None)][m]))),
                                                                   np.exp(np.nanmean(np.log(FF[(area, 1.32)][m])))))

**Result: fig3d-v1-fano-closest reproduced.** The V1 geometric mean Fano factor is 1.051 for
targets alone and 0.985 at 1.3 degrees, within 1.05 +/- 0.02 and 0.98 +/- 0.01; paired t test
p = 3.3e-08 on the Fano factors and 1.4e-07 on their logs.

**Result: fig3e-v4-fano-all-spacings reproduced.** In V4 the values are 0.997 alone and 0.928 at
1.3 degrees, within 1.00 +/- 0.01 and 0.93 +/- 0.01, and the decrease has p < 0.001 at each of
the four spacings under both tests (largest p 1.4e-09 on the Fano factors, 8.7e-09 on logs).

## Modulation indices (Figure 4)

> fig4a-v1-modulation-indices: In V1, distractors at the closest spacing suppress and facilitate
> responses in roughly equal proportion, with significant suppression in 23% and facilitation in
> 27% of neurons; for farther distractors modulation is weaker, with 2.5% significant facilitation
> and 13% suppression.
>
> fig4b-v4-modulation-indices: For the closest distractors most V4 neurons are facilitated (mean
> index 1.11 +/- 0.38, t test, p < 0.001; significant facilitation in 51% and suppression in 14% of
> cases), and the average index is larger than in V1 (one-sided t test, p < 0.01); at larger
> spacing facilitation and suppression are roughly equal (0.97 +/- 0.48; 24% and 32% significant).

Index per neuron and spacing: geometric mean over the four target orientations of the crowded
over the target-alone mean count. Null: target-alone and crowded responses combined, resampled
with replacement in the experimental numbers, 1000 times (STAR Methods); combined within each
orientation or over all orientations. Tail rules: (a) two-sided, 2.5% per tail; (b) one-sided,
5% per direction. Neurons with a zero target-alone or crowded mean at some orientation have no
index and are dropped.

In [ ]:
rng = np.random.default_rng(4)
MI, NULL = {}, {}
for area in AREAS:
    for s in SP:
        MI[(area, s)] = np.concatenate([cw.modulation(p, s) for p in pops(area)])
        for pool in ("orientation", "all"):
            NULL[(area, s, pool)] = np.concatenate([cw.modulation_null(p, s, rng, pool=pool) for p in pops(area)])


def sig_counts(area, s, pool, rule):
    mi, nl = MI[(area, s)], NULL[(area, s, pool)]
    ok = np.isfinite(mi)
    # rank among the null draws that are defined (a draw is undefined when a resampled mean is 0)
    rank = np.sum(nl < mi[:, None], 1) / np.maximum(np.sum(np.isfinite(nl), 1), 1)
    tail = 0.025 if rule == "two-sided" else 0.05
    fac = ok & (rank > 1 - tail)
    sup = ok & (rank < tail)
    return ok, fac, sup


rows = []
for area in AREAS:
    for s in SP:
        for pool in ("orientation", "all"):
            for rule in ("two-sided", "one-sided"):
                ok, fac, sup = sig_counts(area, s, pool, rule)
                mi = MI[(area, s)][ok]
                rows.append(dict(area=area, spacing=SPL[s], pool=pool, rule=rule, n=int(ok.sum()),
                                 dropped=int((~ok).sum()), frac_above_1=np.mean(mi > 1), frac_below_1=np.mean(mi < 1),
                                 sig_fac_pct_all=100 * fac.sum() / ok.sum(), sig_sup_pct_all=100 * sup.sum() / ok.sum(),
                                 sig_fac_pct_of_facilitated=100 * fac.sum() / (mi > 1).sum(),
                                 sig_sup_pct_of_suppressed=100 * sup.sum() / (mi < 1).sum()))
        # farther spacings pooled (reading b)
for area in AREAS:
    for pool in ("orientation", "all"):
        for rule in ("two-sided", "one-sided"):
            oks, facs, sups, mis = [], [], [], []
            for s in SP[1:]:
                ok, fac, sup = sig_counts(area, s, pool, rule)
                oks.append(ok); facs.append(fac); sups.append(sup); mis.append(MI[(area, s)])
            ok, fac, sup, mi = map(np.concatenate, (oks, facs, sups, mis))
            mi = mi[ok]
            rows.append(dict(area=area, spacing="1.6-2.6 pooled", pool=pool, rule=rule, n=int(ok.sum()),
                             dropped=int((~ok).sum()), frac_above_1=np.mean(mi > 1), frac_below_1=np.mean(mi < 1),
                             sig_fac_pct_all=100 * fac.sum() / ok.sum(), sig_sup_pct_all=100 * sup.sum() / ok.sum(),
                             sig_fac_pct_of_facilitated=100 * fac.sum() / (mi > 1).sum(),
                             sig_sup_pct_of_suppressed=100 * sup.sum() / (mi < 1).sum()))
mod = pd.DataFrame(rows)
print(mod.round(1).to_string())
print("Undefined null draws (a resampled mean of 0), percent of all draws of the neurons with a defined index:")
for area in AREAS:
    for s in SP:
        ok = np.isfinite(MI[(area, s)])
        print(f"   {area} {SPL[s]} deg: " + ", ".join(
            f"pool {pool} {100 * np.mean(~np.isfinite(NULL[(area, s, pool)][ok])):.2f}%" for pool in ("orientation", "all")))

In [ ]:
print("V1 expected: closest 23 +/- 2.3 % suppression and 27 +/- 2.7 % facilitation; farther 2.5 +/- 0.25 % "
      "facilitation and 13 +/- 1.3 % suppression (percent of all neurons)")
for pool in ("orientation", "all"):
    for rule in ("two-sided", "one-sided"):
        near = mod.query("area=='V1' and spacing=='1.3' and pool==@pool and rule==@rule").iloc[0]
        ok_near = abs(near.sig_sup_pct_all - 23) <= 2.3 and abs(near.sig_fac_pct_all - 27) <= 2.7
        for far_label in ("2.6", "1.6-2.6 pooled"):
            far = mod.query("area=='V1' and spacing==@far_label and pool==@pool and rule==@rule").iloc[0]
            ok_far = abs(far.sig_fac_pct_all - 2.5) <= 0.25 and abs(far.sig_sup_pct_all - 13) <= 1.3
            print(f"pool {pool:11s} {rule:9s} far={far_label:15s} near sup {near.sig_sup_pct_all:5.1f} fac "
                  f"{near.sig_fac_pct_all:5.1f} (within {ok_near}); far fac {far.sig_fac_pct_all:5.1f} sup "
                  f"{far.sig_sup_pct_all:5.1f} (within {ok_far})")
mi1 = MI[("V1", 1.32)]
print("V1 closest spacing: index above 1 in %.1f%%, below 1 in %.1f%% of neurons; median %.3f"
      % (100 * np.nanmean(mi1 > 1), 100 * np.nanmean(mi1 < 1), np.nanmedian(mi1)))

**Result: fig4a-v1-modulation-indices partially reproduced.** At 1.3 degrees, with the two-sided
rule, 24.5% of V1 neurons are significantly suppressed and 26.5% facilitated (pooling within
orientation; 24.7% and 26.0% pooling over all), within 23 +/- 2.3% and 27 +/- 2.7%; 56.6% have an
index below 1 and 43.4% above. At farther spacings no reading puts both values within tolerance:
at 2.6 degrees, two-sided, facilitation is 2.0% or 1.8% (expected 2.5 +/- 0.25) with suppression
14.4% or 12.9% (13 +/- 1.3); one-sided, facilitation is 2.5% or 2.3% but suppression 19.7% or
20.2%; pooled over the three farther spacings facilitation is 4.0 to 5.1%. The modulation is
weaker at farther spacings, as the claim says, and the far percentages are near the paper's. At
2.6 degrees, two-sided and pooled within orientation, 2.0% is 8 of 396 neurons and 14.4% is 57;
the tolerances (2.25 to 2.75% and 11.7 to 14.3%) correspond to 9 or 10 and 47 to 56 neurons, so
each far value misses by one neuron. The facilitation tolerance of 0.25 points is about
one neuron of 396, finer than the counts can resolve; this is asked of a reader in
`questions/henry-2022.yaml`, and the claim is judged as the claims file has it.

In [ ]:
mi4, mi1 = MI[("V4", 1.32)], MI[("V1", 1.32)]
v4, v1 = mi4[np.isfinite(mi4)], mi1[np.isfinite(mi1)]
mi4f = np.concatenate([MI[("V4", s)] for s in SP[1:]])
mi4f = mi4f[np.isfinite(mi4f)]
mi4_26 = MI[("V4", 2.56)][np.isfinite(MI[("V4", 2.56)])]
print(f"V4 closest: mean index {v4.mean():.3f}, SD {v4.std(ddof=1):.3f}, SEM {v4.std(ddof=1) / np.sqrt(len(v4)):.3f}, "
      f"n = {len(v4)}; above 1 in {100 * np.mean(v4 > 1):.1f}%")
print("   t test against 1 on the index: p = %.2g; on log index: p = %.2g"
      % (stats.ttest_1samp(v4, 1).pvalue, stats.ttest_1samp(np.log(v4), 0).pvalue))
print("   V4 > V1, one-sided two-sample t test on the index: p = %.2g (Welch %.2g); on log index: p = %.2g (Welch %.2g)"
      % (ttest_1s_greater(v4, v1).pvalue, ttest_1s_greater(v4, v1, False).pvalue,
         ttest_1s_greater(np.log(v4), np.log(v1)).pvalue, ttest_1s_greater(np.log(v4), np.log(v1), False).pvalue))
print(f"V1 closest: mean index {v1.mean():.3f}, SD {v1.std(ddof=1):.3f}")
print(f"V4 farthest (2.6): mean {mi4_26.mean():.3f}, SD {mi4_26.std(ddof=1):.3f}; three larger spacings pooled: "
      f"mean {mi4f.mean():.3f}, SD {mi4f.std(ddof=1):.3f}")
print("Expected: V4 closest 1.11 +/- 0.111, facilitation 51 +/- 5.1 %, suppression 14 +/- 1.4 %; "
      "larger spacing 0.97 +/- 0.097, facilitation 24 +/- 2.4 %, suppression 32 +/- 3.2 %")
for pool in ("orientation", "all"):
    for rule in ("two-sided", "one-sided"):
        near = mod.query("area=='V4' and spacing=='1.3' and pool==@pool and rule==@rule").iloc[0]
        for far_label, mfar in (("2.6", mi4_26), ("1.6-2.6 pooled", mi4f)):
            far = mod.query("area=='V4' and spacing==@far_label and pool==@pool and rule==@rule").iloc[0]
            for denom in ("all", "of"):
                if denom == "all":
                    nf, ns, ff_, fs = near.sig_fac_pct_all, near.sig_sup_pct_all, far.sig_fac_pct_all, far.sig_sup_pct_all
                else:
                    nf, ns = near.sig_fac_pct_of_facilitated, near.sig_sup_pct_of_suppressed
                    ff_, fs = far.sig_fac_pct_of_facilitated, far.sig_sup_pct_of_suppressed
                vals_ok = [abs(v4.mean() - 1.11) <= 0.111, abs(nf - 51) <= 5.1, abs(ns - 14) <= 1.4,
                           abs(mfar.mean() - 0.97) <= 0.097, abs(ff_ - 24) <= 2.4, abs(fs - 32) <= 3.2]
                print(f"pool {pool:11s} {rule:9s} far={far_label:15s} denominator={denom:3s}: near fac {nf:5.1f} sup {ns:5.1f}; "
                      f"far fac {ff_:5.1f} sup {fs:5.1f}; values within: {vals_ok}")

**Result: fig4b-v4-modulation-indices reproduced, under the two-sided rule with percentages of
all neurons, the farthest spacing as "larger spacing", and the t tests on log indices.** The V4
mean index at 1.3 degrees is 1.134 (SD 0.379, printed in the paper as +/- 0.38), with 69.8% of
neurons above 1 and p = 1.9e-19 against 1; 50.7% are significantly facilitated and 15.1% (or
14.3%) suppressed. At 2.6 degrees the mean is 1.009 (SD 0.473), with 23.9% (23.5%) facilitated
and 33.8% suppressed. All six values are within tolerance under both poolings. V4 against V1 at
1.3 degrees, one-sided: p = 0.009 on log indices, below 0.01, but p = 0.074 on the indices
themselves (V1 mean 1.097), so this part rests on the log reading. With percentages of the
facilitated and suppressed neurons as denominators, the values are far out (72.6% and 49.8%).

## Modulation strength (Figure 4C)

> fig4c-modulation-strength-significant: The strength of modulation, the mean absolute log
> modulation index, is significant in both areas at all spacings (permutation test, p < 0.01).
>
> fig4c-v4-stronger-than-v1: Modulation strength is stronger in V4 than in V1 at all separations
> (one-sided t test, p = 0.01 for the closest distractors, p < 0.001 for the others).
>
> fig4c-v4-maintained-v1-weaker-far: Modulation strength in V4 is largely maintained across
> spacings (one-sided t test nearest against farthest, p = 0.44), whereas in V1 the farthest
> separation yields weaker modulation than the nearest (one-sided t test, p < 0.001).

Strength per neuron is |log2 index|. The null is the per-neuron resampled indices above (both
poolings), averaged over neurons in each of the 1000 resamples.

In [ ]:
STR = {k: np.abs(np.log2(v)) for k, v in MI.items()}
rows = []
for area in AREAS:
    for s in SP:
        x = STR[(area, s)]
        obs = np.nanmean(x)
        r = dict(area=area, spacing=SPL[s], n=int(np.isfinite(x).sum()), strength=obs,
                 sem=np.nanstd(x, ddof=1) / np.sqrt(np.isfinite(x).sum()))
        for pool in ("orientation", "all"):
            nl = np.nanmean(np.abs(np.log2(NULL[(area, s, pool)][np.isfinite(x)])), 0)
            r[f"null_mean_{pool}"] = nl.mean()
            r[f"null_99pct_{pool}"] = np.percentile(nl, 99)
            r[f"p_{pool}"] = (1 + np.sum(nl >= obs)) / (1 + len(nl))
        rows.append(r)
ms = pd.DataFrame(rows)
print(ms.to_string(float_format=lambda v: f"{v:.4g}"))
print("Observed above the 99th null percentile at all eight: pooling within orientation",
      bool((ms.strength > ms.null_99pct_orientation).all()), "; over all orientations",
      bool((ms.strength > ms.null_99pct_all).all()))

**Result: fig4c-modulation-strength-significant reproduced.** In all eight area and spacing pairs
the observed strength (0.087 to 0.285) exceeds the 99th percentile of its null (0.071 to 0.074),
under both poolings; p = 0.001, the smallest attainable with 1000 resamples.

In [ ]:
for s in SP:
    a, b = STR[("V4", s)], STR[("V1", s)]
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    print(f"{SPL[s]} deg: V4 {a.mean():.4f} vs V1 {b.mean():.4f}; one-sided Student p = {ttest_1s_greater(a, b).pvalue:.2g}, "
          f"Welch p = {ttest_1s_greater(a, b, False).pvalue:.2g}")

**Result: fig4c-v4-stronger-than-v1 reproduced.** V4 strength exceeds V1 at every spacing:
one-sided p = 0.011 at 1.3 degrees (paper 0.01; Welch 0.012) and p from 1.9e-17 to 4.7e-26 at
the other three (Welch 2.8e-22 to 1.4e-39).

In [ ]:
for area in AREAS:
    n_, f_ = STR[(area, 1.32)], STR[(area, 2.56)]
    ok = np.isfinite(n_) & np.isfinite(f_)
    tp = stats.ttest_rel(n_[ok], f_[ok], alternative="greater")
    ti = stats.ttest_ind(n_[np.isfinite(n_)], f_[np.isfinite(f_)], alternative="greater")
    print(f"{area}: nearest {np.nanmean(n_):.4f}, farthest {np.nanmean(f_):.4f}, difference {np.nanmean(n_) - np.nanmean(f_):.4f}; "
          f"one-sided paired p = {tp.pvalue:.3g} (n = {ok.sum()}), one-sided two-sample p = {ti.pvalue:.3g}")

**Result: fig4c-v4-maintained-v1-weaker-far reproduced.** In V4, nearest 0.2854 against farthest
0.2813 (difference 0.0041): one-sided p = 0.403 paired and 0.409 two-sample, against the paper's
0.44, so neither test finds a decrease. In V1, 0.2426 against 0.0866: p = 9.3e-23 paired and
2.5e-22 two-sample, below 0.001.

## Tuning model (Figure 5, Figure S5, fit quality)

> fig5bc-tuning-model-terms: With both terms allowed, V1 tuning amid distractors has a
> multiplicative term below 1 (0.59 +/- 0.03) and a positive additive term (0.39 +/- 0.16); in V4
> the multiplicative term is 0.36 +/- 0.01 and the additive term 0.90 +/- 0.02; the
> multiplicative term is smaller and the additive term larger in V4 than in V1 (t test).

Neurons whose target-alone tuning from even and from odd presentations of each orientation (in
stored order) correlates above 0.5. In V4 the stored order is not the recording order (data
check), so there the split is in effect a random half split. Per neuron and spacing, rTD =
max(m*rT + a*mean(rT), 0), m and a by maximum Poisson likelihood, on single-trial counts (reading
1a) and on mean counts per orientation (1b).

In [ ]:
REL = {area: np.concatenate([cw.even_odd_reliability(p) for p in pops(area)]) for area in AREAS}
for area in AREAS:
    print(f"{area}: reliability > 0.5 in {np.sum(REL[area] > 0.5)} of {len(REL[area])} neurons "
          f"(undefined in {np.sum(~np.isfinite(REL[area]))}); per animal "
          f"{ {an: int(np.sum((REL[area] > 0.5) & (animal[area] == an))) for an in ('AL', 'CA')} }")
# The V4 stored order is not the recording order (data check). Two order-free variants: the count
# passing under single random half splits (what an even/odd split of a random order gives), and
# REL_R, the reliability averaged over 20 random half splits, whose selection is fitted below too.
rng_sp = np.random.default_rng(13)
for area in AREAS:
    cnt = [sum(int(np.sum(cw.random_split_reliability(p, rng_sp, 1) > 0.5)) for p in pops(area)) for _ in range(20)]
    print(f"{area}: neurons above 0.5 under 20 single random half splits: mean {np.mean(cnt):.1f}, range {min(cnt)} to {max(cnt)}")
rng = np.random.default_rng(6)
REL_R = {area: np.concatenate([cw.random_split_reliability(p, rng) for p in pops(area)]) for area in AREAS}
for area in AREAS:
    print(f"{area}: REL_R (mean over 20 random splits) > 0.5 in {np.sum(REL_R[area] > 0.5)} neurons; per animal "
          f"{ {an: int(np.sum((REL_R[area] > 0.5) & (animal[area] == an))) for an in ('AL', 'CA')} }; "
          f"both selections {np.sum((REL[area] > 0.5) & (REL_R[area] > 0.5))}")


def fit_population(p, rel, single_trial):
    rows = []
    rT_all = cw.tuning(p, "alone")
    for i in np.flatnonzero(rel > 0.5):
        rT = rT_all[i]
        for s in SP:
            rc = [r[i] for r in cw.responses(p, "crowd", s)]
            m, a, ll = cw.fit_affine(rT, rc, single_trial)
            pred = np.maximum(m * rT + a * rT.mean(), cw.EPS)
            ll_null = cw._poisson_ll(np.full(len(rT), np.mean(np.concatenate(rc))), rc, True)
            ll_orac = cw._poisson_ll(np.array([c.mean() for c in rc]), rc, True)
            ll_mod = cw._poisson_ll(pred, rc, True)
            cm, llm = cw.fit_single(rT, rc, "mult", single_trial)
            ca, lla = cw.fit_single(rT, rc, "add", single_trial)
            rows.append(dict(area=p["area"], animal=p["animal"], session=p["session"], orange=p["orange"],
                             neuron=i, spacing=s, m=m, a=a, ll_null=ll_null, ll_oracle=ll_orac, ll_model=ll_mod,
                             c_mult=cm, c_add=ca, ll_mult=llm, ll_add=lla,
                             n_trials=[len(c) for c in rc]))
    return rows


t0 = time.time()
FIT = {}
for st in (True, False):
    res = Parallel(n_jobs=4)(delayed(fit_population)(p, cw.even_odd_reliability(p), st) for p in S)
    FIT[st] = pd.DataFrame([r for rr in res for r in rr])
offsets = {area: np.cumsum([0] + [p["counts"].shape[0] for p in pops(area)]) for area in AREAS}
rel_r_pop = [REL_R[p["area"]][offsets[p["area"]][k]:offsets[p["area"]][k + 1]]
             for area in AREAS for k, p in enumerate(pops(area))]
pop_order = [p for area in AREAS for p in pops(area)]
FIT_R = pd.DataFrame([r for rr in Parallel(n_jobs=4)(delayed(fit_population)(p, rl, True)
                                                     for p, rl in zip(pop_order, rel_r_pop)) for r in rr])
print(f"fits done in {time.time() - t0:.0f} s; cases per area:", FIT[True].groupby("area").size().to_dict(),
      "; random-split selection (REL_R):", FIT_R.groupby("area").size().to_dict())

rows = []
for st in (True, False):
    f = FIT[st]
    for unit in ("cases", "neurons"):
        g = f if unit == "cases" else f.groupby(["area", "session", "orange", "neuron"], as_index=False)[["m", "a"]].mean()
        r = dict(likelihood="single trial" if st else "means", unit=unit)
        for area in AREAS:
            x = g[g.area == area]
            r[f"{area}_n"] = len(x)
            r[f"{area}_m"] = x.m.mean(); r[f"{area}_m_sem"] = x.m.std(ddof=1) / np.sqrt(len(x))
            r[f"{area}_a"] = x.a.mean(); r[f"{area}_a_sem"] = x.a.std(ddof=1) / np.sqrt(len(x))
            r[f"{area}_m_below1_p"] = stats.ttest_1samp(x.m, 1, alternative="less").pvalue
        a1, a4 = g[g.area == "V1"], g[g.area == "V4"]
        r["m_V4_lt_V1_p_two_sided"] = stats.ttest_ind(a4.m, a1.m).pvalue
        r["m_V4_minus_V1"] = a4.m.mean() - a1.m.mean()
        r["a_V4_gt_V1_p_two_sided"] = stats.ttest_ind(a4.a, a1.a).pvalue
        r["a_V4_minus_V1"] = a4.a.mean() - a1.a.mean()
        rows.append(r)
f5 = pd.DataFrame(rows)
print(f5.T.to_string(float_format=lambda v: f"{v:.4g}"))
print()
print("Expected: V1 m 0.59 +/- 0.03, a 0.39 +/- 0.16; V4 m 0.36 +/- 0.01, a 0.90 +/- 0.02; 146 and 198 neurons")
for _, r in f5.iterrows():
    ok = [abs(r.V1_m - 0.59) <= 0.03, abs(r.V1_a - 0.39) <= 0.16, abs(r.V4_m - 0.36) <= 0.01, abs(r.V4_a - 0.90) <= 0.02]
    print(f"{r.likelihood:12s} {r.unit:8s} values within: {ok}")
f = FIT[True]
print("Range of fitted m: V1", np.round([f[f.area == 'V1'].m.min(), f[f.area == 'V1'].m.max()], 2),
      "V4", np.round([f[f.area == 'V4'].m.min(), f[f.area == 'V4'].m.max()], 2),
      "; of a: V1", np.round([f[f.area == 'V1'].a.min(), f[f.area == 'V1'].a.max()], 2),
      "V4", np.round([f[f.area == 'V4'].a.min(), f[f.area == 'V4'].a.max()], 2))
print("Median m and a (single trial, cases):", f.groupby("area")[["m", "a"]].median().round(3).to_dict())
print("Per animal (single trial, cases):")
print(f.groupby(["area", "animal"]).agg(neurons=("neuron", lambda x: len(x) // 4), m=("m", "mean"), a=("a", "mean")).round(3).to_string())
fig, axs = plt.subplots(1, 2, figsize=(8, 3.6))
for ax, area in zip(axs, AREAS):
    x = f[f.area == area]
    ax.scatter(x.m, x.a, c=x.spacing, cmap="viridis_r", s=6)
    ax.set(xlim=(-2, 2), ylim=(-2, 2), xlabel="multiplicative term", ylabel="additive term", title=area)
plt.tight_layout()
plt.show()
for area in AREAS:
    x = f[f.area == area]
    print(area, "cases outside the axes (|m| or |a| > 2):", int(((x.m.abs() > 2) | (x.a.abs() > 2)).sum()), "of", len(x))

**Result: partially reproduced.** The even/odd criterion keeps 146 V1 neurons, exactly the
paper's count (89 and 57 per animal, as in Figure S7), and 244 V4 neurons against 198 +/- 20 (114
and 130 against 93 and 105). The V4 epochs are not stored in recording order (data check), but
the order does not explain the excess: single random half splits keep 238.3 V4 neurons on average
(219 to 256 over 20 splits) and 143.8 V1 neurons (126 to 163), so both stored-order counts lie in
the range of random splits. The directions hold: m is below 1 in both areas (p < 1e-35), V4 has
the smaller m (difference -0.251, p = 3.6e-18) and the larger a (+0.312, p = 3.7e-18). The values
do not: over cases V1 m = 0.405 +/- 0.023 and a = 0.596 +/- 0.026 (expected 0.59 and 0.39), V4 m
= 0.154 +/- 0.017 (expected 0.36) and a = 0.907 +/- 0.023, the only one within tolerance. Fits on
means and averages over neurons change these by less than 0.001. On the selection by reliability
averaged over 20 random splits (109 V1 and 160 V4 neurons, below) the picture is the same: m 0.403
and 0.178, a 0.588 and 0.894, only the V4 additive term within tolerance, and the V4 differences
at p = 4.1e-16 and 1.9e-14.


## fit-quality

> In the neurons fitted with the tuning model, the fit quality (a normalized likelihood between a
> null model at 0 and an oracle at 1) is 0.63 in V1 and 0.73 in V4.

In [ ]:
for st in (True, False):
    f = FIT[st].copy()
    f["q"] = (f.ll_model - f.ll_null) / (f.ll_oracle - f.ll_null)
    ok = (f.ll_oracle - f.ll_null) > 1e-9
    for area in AREAS:
        x = f[ok & (f.area == area)]
        pooled = (x.ll_model - x.ll_null).sum() / (x.ll_oracle - x.ll_null).sum()
        print(f"fits on {'single trials' if st else 'means'}: {area} mean per case {x.q.mean():.3f}, median per case "
              f"{x.q.median():.3f}, pooled {pooled:.3f} (n = {len(x)}, degenerate cases dropped {int((~ok & (f.area == area)).sum())}; "
              f"range {x.q.min():.2f} to {x.q.max():.2f})")
print("Expected: V1 0.63 +/- 0.063, V4 0.73 +/- 0.073")

**Result: not reproduced.** The V1 fit quality is 0.623 when pooled over cases, within 0.63 +/-
0.063, but 0.455 as a mean and 0.460 as a median per case; V4 gives 0.589 pooled, 0.410 mean and
0.328 median, all outside 0.73 +/- 0.073. No reading puts both values within tolerance. On the random-split selection (below) V1 gives 0.489
mean, 0.494 median and 0.663 pooled, and V4 0.440, 0.410 and 0.651, again with V4 outside
tolerance under every reading.

## fig5-terms-covary

> In both V1 and V4 the additive and multiplicative terms covary strongly, consistent with a
> flattening of tuning with maintained responsivity.

In [ ]:
for st in (True, False):
    f = FIT[st]
    for area in AREAS:
        x = f[f.area == area]
        r, p = stats.pearsonr(x.m, x.a)
        rs, ps = stats.spearmanr(x.m, x.a)
        print(f"fits on {'single trials' if st else 'means'}: {area} Pearson r = {r:.3f} (p = {p:.2g}), "
              f"Spearman rho = {rs:.3f} (p = {ps:.2g}), n = {len(x)} cases")

**Result: reproduced.** m and a are strongly and negatively correlated: Pearson r = -0.892 in V1
(584 cases) and -0.682 in V4 (976 cases), p below 1e-130 (Spearman -0.955 and -0.839). On the random-split selection (below) V1 gives r =
-0.913 but V4 r = -0.459 (p = 1.1e-34), below the criterion's 0.5, which would make the outcome
partial. The outcome therefore rests on the even/odd selection, whose counts match those of
single random splits (fig5bc), and is marked as resting on that assumption.

## fig5-negative-multiplicative

> Some fits give a negative multiplicative term, offset by a large additive term; such cases are
> rare in V1 and more frequent in V4.

In [ ]:
for st in (True, False):
    f = FIT[st]
    tab = []
    for area in AREAS:
        x = f[f.area == area]
        neg = x.m < 0
        tab.append([neg.sum(), (~neg).sum()])
        print(f"fits on {'single trials' if st else 'means'}: {area} m < 0 in {neg.sum()} of {len(x)} cases "
              f"({100 * neg.mean():.1f}%); mean a with m < 0 {x.a[neg].mean():.3f}, otherwise {x.a[~neg].mean():.3f}")
    print("   Fisher exact test, two-sided: p = %.2g" % stats.fisher_exact(tab)[1])

**Result: partially reproduced.** Negative multiplicative terms are more frequent in V4 (35.1% of
cases) than in V1 (23.3%; Fisher p = 7.7e-07), and they come with large additive terms (mean a
1.29 and 1.41 against 0.38 and 0.63). But 23.3% of V1 cases is not rare by the criterion's
reading (below 10%). On the random-split selection (below) the fractions are 20.2% and 33.6% (Fisher p =
1.3e-06), the same partial outcome.


## fig5de-simulated-discriminability

> Simulating crowded responses with the fitted multiplicative and additive terms yields some loss
> of discriminability in V1 and more in V4, similar to but slightly stronger than the measured
> loss, and depending weakly on spacing.

For each fitted neuron and spacing, Poisson counts at the two extreme orientations with the
measured target-alone means and with the model's prediction (single-trial fits); AUC with the
preferred extreme as in Figure 2; OLS slope with 1000 bootstrap samples. Trials per stimulus:
(a) as recorded, (b) 1000. Measured slopes: (a) Figure 2C over all neurons, (b) the fitted
neurons only.

In [ ]:
rng = np.random.default_rng(5)
popkey = {(p["area"], p["session"], p["orange"]): p for p in S}


def simulate_aucs(f, mode, rng, rule="affine"):
    """Simulated (alone, crowded) AUCs per case. rule 'affine' uses m and a; 'single' the
    additive fit when the case's index is above 1 and the multiplicative one otherwise."""
    out = []
    for _, r in f.iterrows():
        p = popkey[(r.area, r.session, r.orange)]
        first, second = cw.extreme_order(p)
        rT = cw.tuning(p, "alone")[r.neuron]
        if rule == "affine":
            pred = np.maximum(r.m * rT + r.a * rT.mean(), 0)
        elif r.mi > 1:
            pred = np.maximum(rT + r.c_add * rT.mean(), 0)
        else:
            pred = np.maximum(r.c_mult * rT, 0)
        ra, rc = cw.responses(p, "alone"), cw.responses(p, "crowd", r.spacing)
        if mode == "recorded":
            na1, na2, nc1, nc2 = ra[first].shape[1], ra[second].shape[1], rc[first].shape[1], rc[second].shape[1]
        else:
            na1 = na2 = nc1 = nc2 = 1000
        a = cw.auc(rng.poisson(rT[first], na1), rng.poisson(rT[second], na2))
        c = cw.auc(rng.poisson(pred[first], nc1), rng.poisson(pred[second], nc2))
        out.append((a, c))
    return np.array(out)


fitted = FIT[True].copy()
fitted["mi"] = [cw.modulation(popkey[(r.area, r.session, r.orange)], r.spacing)[r.neuron] for _, r in fitted.iterrows()]
meas_fitted = {}
for area in AREAS:
    for s in SP:
        x = fitted[(fitted.area == area) & (fitted.spacing == s)]
        a = np.array([cw.neuron_aucs(popkey[(r.area, r.session, r.orange)], "alone")[r.neuron] for _, r in x.iterrows()])
        c = np.array([cw.neuron_aucs(popkey[(r.area, r.session, r.orange)], "crowd", s)[r.neuron] for _, r in x.iterrows()])
        meas_fitted[(area, s)] = cw.ols_slope(a, c)
SIM = {}
rows = []
for rule in ("affine", "single"):
    for mode in ("recorded", "1000"):
        for area in AREAS:
            for s in SP:
                x = fitted[(fitted.area == area) & (fitted.spacing == s)]
                ac = simulate_aucs(x, mode, rng, rule)
                sl, lo, hi = cw.slope_ci(ac[:, 0], ac[:, 1], rng)
                m2 = f2[(f2.area == area) & (f2.spacing == SPL[s])].iloc[0].slope
                rows.append(dict(model=rule, trials=mode, area=area, spacing=SPL[s], n=len(x), sim_slope=sl, ci_low=lo,
                                 ci_high=hi, measured_all=m2, measured_fitted=meas_fitted[(area, s)]))
sim = pd.DataFrame(rows)
print(sim.round(3).to_string())
print()
for mode in ("recorded", "1000"):
    d = sim[(sim.model == "affine") & (sim.trials == mode)]
    v1, v4 = d[d.area == "V1"].sim_slope.values, d[d.area == "V4"].sim_slope.values
    print(f"affine, trials {mode}: V4 simulated slope below V1 at every spacing: {bool(np.all(v4 < v1))}")
    for meas in ("measured_all", "measured_fitted"):
        for area in AREAS:
            dd = d[d.area == area]
            print(f"   {meas:16s} {area}: simulated at or below measured at {int((dd.sim_slope <= dd[meas]).sum())} of 4 spacings")

**Result: reproduced, under reading (a) of both open details (recorded trial counts, Figure 2C
slopes over all neurons).** The simulated V4 slope is below the V1 slope at every spacing (0.243
to 0.336 against 0.404 to 0.567), and the simulated slope is at or below the measured one at 4 of
4 spacings in V1 and 3 of 4 in V4 (V4 at 1.3 degrees: 0.243 against 0.206). With 1000 trials per
stimulus V4 is at or below the measured slope at only 1 of 4 spacings over all neurons (3 of 4
against the fitted neurons' own slopes). On the random-split selection (below) the simulated V4
slope is at or below the measured one at only 1 of 4 spacings (V1 4 of 4), so the outcome rests on
the even/odd selection, whose counts match those of single random splits (fig5bc), and is marked
as resting on that assumption.

## figs5-additive-vs-multiplicative

> Fitting either a multiplicative or an additive change of target tuning alone, neurons
> facilitated by distractors are better described by the additive model and suppressed neurons by
> the multiplicative model, in V1 and V4; each single form reduces discriminability, but less than
> in the measured responses.

Single-form fits on the same cases (the tuning model's neuron selection); log-likelihood ratio,
multiplicative over additive, against the case's modulation index. Panels C and D: the simulation
above with the additive fit for facilitated cases and the multiplicative fit for suppressed ones
(rows `model == single`). The all-driven-neurons variant of the selection is shown below.

In [ ]:
for st in (True, False):
    f = FIT[st].copy()
    f["mi"] = fitted["mi"].values
    f["llr"] = f.ll_mult - f.ll_add
    for area in AREAS:
        x = f[f.area == area]
        fac, sup = x[x.mi > 1], x[x.mi < 1]
        print(f"fits on {'single trials' if st else 'means'}: {area} facilitated cases favouring additive (LLR < 0): "
              f"{(fac.llr < 0).sum()} of {len(fac)} ({100 * (fac.llr < 0).mean():.1f}%); suppressed favouring "
              f"multiplicative (LLR > 0): {(sup.llr > 0).sum()} of {len(sup)} ({100 * (sup.llr > 0).mean():.1f}%)")


def single_form_all(p):
    rows = []
    rT_all = cw.tuning(p, "alone")
    for i in range(p["counts"].shape[0]):
        mi = {s: cw.modulation(p, s)[i] for s in SP}
        for s in SP:
            rc = [r[i] for r in cw.responses(p, "crowd", s)]
            _, llm = cw.fit_single(rT_all[i], rc, "mult")
            _, lla = cw.fit_single(rT_all[i], rc, "add")
            rows.append(dict(area=p["area"], mi=mi[s], llr=llm - lla))
    return rows


allf = pd.DataFrame([r for rr in Parallel(n_jobs=4)(delayed(single_form_all)(p) for p in S) for r in rr])
for area in AREAS:
    x = allf[(allf.area == area) & np.isfinite(allf.mi)]
    fac, sup = x[x.mi > 1], x[x.mi < 1]
    print(f"all neurons: {area} facilitated favouring additive {100 * (fac.llr < 0).mean():.1f}% of {len(fac)}; "
          f"suppressed favouring multiplicative {100 * (sup.llr > 0).mean():.1f}% of {len(sup)}")
print()
for mode in ("recorded", "1000"):
    d = sim[(sim.model == "single") & (sim.trials == mode)]
    for area in AREAS:
        dd = d[d.area == area]
        print(f"single-form simulation, trials {mode}, {area}: slopes {np.round(dd.sim_slope.values, 3)}; all below 1: "
              f"{bool((dd.sim_slope < 1).all())} (point slopes), {bool((dd.ci_high < 1).all())} (upper CI); above measured (all neurons) at {int((dd.sim_slope > dd.measured_all).sum())} of 4, "
              f"above measured (fitted neurons) at {int((dd.sim_slope > dd.measured_fitted).sum())} of 4")

**Result: reproduced.** Among the tuning model's cases, 89.8% of facilitated V1 cases and 97.0% of
facilitated V4 cases favour the additive model, and 83.7% and 92.1% of suppressed cases favour the
multiplicative one; over all neurons 86.1%, 92.2%, 81.0% and 88.5%. The single-form simulation
(additive for facilitation, multiplicative for suppression) gives slopes below 1 at every spacing
(point slopes 0.776 to 0.911 with recorded trial counts; judged by the upper CI instead, V1 at 2.6
degrees reaches 1.001), above the measured Figure 2 slopes at 4 of 4 spacings in each area. On the
random-split selection (below) the shares are 93.1%, 98.0%, 84.9% and 93.2% and the single-form
slopes 0.807 to 0.887, above the measured ones at 4 of 4 spacings in each area.


### Tuning-model claims on the random-split selection

The fig5bc, fit-quality, fig5-terms-covary, fig5-negative-multiplicative, fig5de and figs5 tests
repeated on the neurons selected by `REL_R` (reliability averaged over 20 random half splits,
single-trial likelihood, means over cases), since the V4 stored order is not the recording order.


In [ ]:
rng = np.random.default_rng(14)
fr = FIT_R.copy()
fr["q"] = (fr.ll_model - fr.ll_null) / (fr.ll_oracle - fr.ll_null)
fr["mi"] = [cw.modulation(popkey[(r.area, r.session, r.orange)], r.spacing)[r.neuron] for _, r in fr.iterrows()]
fr["llr"] = fr.ll_mult - fr.ll_add
okq = (fr.ll_oracle - fr.ll_null) > 1e-9
for area in AREAS:
    x = fr[fr.area == area]
    xq = fr[okq & (fr.area == area)]
    fac, sup = x[x.mi > 1], x[x.mi < 1]
    print(f"{area}: {len(x) // 4} neurons; m {x.m.mean():.3f} +/- {x.m.std(ddof=1) / np.sqrt(len(x)):.3f}, "
          f"a {x.a.mean():.3f} +/- {x.a.std(ddof=1) / np.sqrt(len(x)):.3f}; m < 1 p = {stats.ttest_1samp(x.m, 1, alternative='less').pvalue:.2g}")
    print(f"   fit quality mean {xq.q.mean():.3f}, median {xq.q.median():.3f}, pooled "
          f"{(xq.ll_model - xq.ll_null).sum() / (xq.ll_oracle - xq.ll_null).sum():.3f}")
    print(f"   m, a Pearson r = {stats.pearsonr(x.m, x.a)[0]:.3f} (p = {stats.pearsonr(x.m, x.a)[1]:.2g}); "
          f"m < 0 in {100 * (x.m < 0).mean():.1f}% of {len(x)} cases")
    print(f"   facilitated favouring additive {100 * (fac.llr < 0).mean():.1f}% of {len(fac)}; suppressed favouring "
          f"multiplicative {100 * (sup.llr > 0).mean():.1f}% of {len(sup)}")
a1, a4 = fr[fr.area == "V1"], fr[fr.area == "V4"]
print(f"V4 - V1: m {a4.m.mean() - a1.m.mean():+.3f} (p = {stats.ttest_ind(a4.m, a1.m).pvalue:.2g}), "
      f"a {a4.a.mean() - a1.a.mean():+.3f} (p = {stats.ttest_ind(a4.a, a1.a).pvalue:.2g})")
print("Expected: V1 m 0.59 +/- 0.03, a 0.39 +/- 0.16; V4 m 0.36 +/- 0.01, a 0.90 +/- 0.02; 146 and 198 neurons; "
      "fit quality 0.63 +/- 0.063 and 0.73 +/- 0.073")
print("   within:", {"V1 m": abs(a1.m.mean() - 0.59) <= 0.03, "V1 a": abs(a1.a.mean() - 0.39) <= 0.16,
                     "V4 m": abs(a4.m.mean() - 0.36) <= 0.01, "V4 a": abs(a4.a.mean() - 0.90) <= 0.02})
tab = [[int((x.m < 0).sum()), int((x.m >= 0).sum())] for x in (a1, a4)]
print(f"Fisher exact test, m < 0 in V4 against V1: p = {stats.fisher_exact(tab)[1]:.2g}")
rows = []
for rule in ("affine", "single"):
    for area in AREAS:
        for s in SP:
            x = fr[(fr.area == area) & (fr.spacing == s)]
            ac = simulate_aucs(x, "recorded", rng, rule)
            sl, lo, hi = cw.slope_ci(ac[:, 0], ac[:, 1], rng)
            m2 = f2[(f2.area == area) & (f2.spacing == SPL[s])].iloc[0].slope
            rows.append(dict(model=rule, area=area, spacing=SPL[s], n=len(x), sim_slope=sl, ci_low=lo, ci_high=hi,
                             measured_all=m2))
simr = pd.DataFrame(rows)
print(simr.round(3).to_string())
d = simr[simr.model == "affine"]
print("affine, recorded trials: V4 simulated slope below V1 at every spacing:",
      bool(np.all(d[d.area == "V4"].sim_slope.values < d[d.area == "V1"].sim_slope.values)),
      "; at or below measured (all neurons):", {a: int((d[d.area == a].sim_slope <= d[d.area == a].measured_all).sum()) for a in AREAS})
d = simr[simr.model == "single"]
print("single form, recorded trials: point slopes below 1:", {a: bool((d[d.area == a].sim_slope < 1).all()) for a in AREAS},
      "; above measured:", {a: int((d[d.area == a].sim_slope > d[d.area == a].measured_all).sum()) for a in AREAS})


## Tuning correlations (Figure 6C to 6E)

> fig6c-tuning-correlation-near: Tuning for targets alone correlates well with tuning for targets
> with nearby distractors in V1 (0.38 +/- 0.03 for cells above a reliability criterion of 0.5),
> and significantly less in V4 (0.22 +/- 0.03; one-sided t test, p < 0.001).
>
> fig6d-tuning-correlation-spacing: Target tuning with near and with far distractors correlates
> less in V4 than V1 (0.17 +/- 0.02 against 0.36 +/- 0.02 at criterion 0.5; one-sided t test,
> p < 0.001).
>
> fig6e-tuning-correlation-configuration: In 4 further sessions, target tuning with two distractor
> configurations whose orientations differ by 20 degrees differs substantially in V4 (correlation
> 0.24 +/- 0.06) and is nearly unaffected in V1 (0.65 +/- 0.06; p < 0.001, one-sided t test).

Reliability as for the tuning model (even and odd presentations in stored order); Pearson
correlation of mean count per target orientation between two conditions; mean +/- SEM over
neurons; V1 greater than V4 by a one-sided two-sample t test (Student, Welch shown too).

In [ ]:
def tuning_pooled(p, seps):
    """Mean count per orientation over the epochs of several spacings."""
    return np.stack([np.concatenate([cw.responses(p, "crowd", s)[o] for s in seps], 1).mean(1)
                     for o in range(len(p["oris"]))], 1)


TC = {}
for area in AREAS:
    ta = np.concatenate([cw.tuning(p, "alone") for p in pops(area)])
    near_a = np.concatenate([cw.tuning(p, "crowd", 1.32) for p in pops(area)])
    near_b = np.concatenate([tuning_pooled(p, SP[:2]) for p in pops(area)])
    far_a = np.concatenate([cw.tuning(p, "crowd", 2.56) for p in pops(area)])
    far_b = np.concatenate([tuning_pooled(p, SP[2:]) for p in pops(area)])
    TC[(area, "6C (a) alone vs 1.3")] = cw.rowcorr(ta, near_a)
    TC[(area, "6C (b) alone vs 1.3+1.6")] = cw.rowcorr(ta, near_b)
    TC[(area, "6D (a) 1.3 vs 2.6")] = cw.rowcorr(near_a, far_a)
    TC[(area, "6D (b) 1.3+1.6 vs 2.1+2.6")] = cw.rowcorr(near_b, far_b)


def compare(key, crit, rel):
    out = {}
    for area in AREAS:
        x = TC[(area, key)][(rel[area] > crit)]
        out[area] = x[np.isfinite(x)]
    t = ttest_1s_greater(out["V1"], out["V4"])
    tw = ttest_1s_greater(out["V1"], out["V4"], False)
    return out, t.pvalue, tw.pvalue


rows = []
for key in ["6C (a) alone vs 1.3", "6C (b) alone vs 1.3+1.6", "6D (a) 1.3 vs 2.6", "6D (b) 1.3+1.6 vs 2.1+2.6"]:
    for crit in (0.5, 0.7, 0.9):
        out, p_, pw = compare(key, crit, REL)
        rows.append(dict(comparison=key, criterion=crit, V1_n=len(out["V1"]), V1_mean=out["V1"].mean(),
                         V1_sem=out["V1"].std(ddof=1) / np.sqrt(len(out["V1"])), V4_n=len(out["V4"]),
                         V4_mean=out["V4"].mean(), V4_sem=out["V4"].std(ddof=1) / np.sqrt(len(out["V4"])),
                         p_V1_gt_V4=p_, p_welch=pw))
f6 = pd.DataFrame(rows)
print(f6.to_string(float_format=lambda v: f"{v:.4g}"))
print("Correlations undefined (flat tuning) at criterion 0.5:",
      {k: int((~np.isfinite(v[REL[k[0]] > 0.5])).sum()) for k, v in TC.items()})
print("Expected 6C: V1 0.38 +/- 0.03, V4 0.22 +/- 0.03; 6D: V1 0.36 +/- 0.02, V4 0.17 +/- 0.02; p < 0.001")
for _, r in f6[f6.criterion == 0.5].iterrows():
    e1, e4, tol = (0.38, 0.22, 0.03) if r.comparison.startswith("6C") else (0.36, 0.17, 0.02)
    print(f"{r.comparison:28s} V1 {r.V1_mean:.3f} within {abs(r.V1_mean - e1) <= tol}; V4 {r.V4_mean:.3f} within "
          f"{abs(r.V4_mean - e4) <= tol}; p {r.p_V1_gt_V4:.2g}")
for an in ("AL", "CA"):
    for key in ["6C (a) alone vs 1.3", "6D (a) 1.3 vs 2.6"]:
        v = {area: np.nanmean(TC[(area, key)][(REL[area] > 0.5) & (animal[area] == an)]) for area in AREAS}
        print(f"{an} {key}: V1 {v['V1']:.3f}, V4 {v['V4']:.3f}")

A random half split gives a reliability that does not depend on the stored order. As the V4 order
is not the recording order, the comparison is repeated with reliability from 20 random half splits
of each orientation's presentations (mean correlation over splits), as a variant (`REL_R`, computed in the fig5bc section).

In [ ]:
# REL_R, the reliability averaged over 20 random half splits, is computed in the fig5bc section.
print("Neurons above 0.5 with random splits: V1", int((REL_R["V1"] > 0.5).sum()), "V4", int((REL_R["V4"] > 0.5).sum()))
for key in ["6C (a) alone vs 1.3", "6D (a) 1.3 vs 2.6"]:
    out, p_, pw = compare(key, 0.5, REL_R)
    print(f"{key}: V1 {out['V1'].mean():.3f} +/- {out['V1'].std(ddof=1) / np.sqrt(len(out['V1'])):.3f} (n {len(out['V1'])}), "
          f"V4 {out['V4'].mean():.3f} +/- {out['V4'].std(ddof=1) / np.sqrt(len(out['V4'])):.3f} (n {len(out['V4'])}), p = {p_:.2g}")

**Result: fig6c-tuning-correlation-near partially reproduced.** At criterion 0.5, reading (a),
V1 gives 0.389 +/- 0.042, within 0.38 +/- 0.03, and V4 0.161 +/- 0.039, below 0.22 +/- 0.03;
reading (b) gives 0.460 (out) and 0.238 (within). V4 is lower than V1 at p = 8.9e-05 (a) and
1.2e-04 (b), and at criteria 0.7 and 0.9 too (p = 5.9e-05 and 0.009). No reading puts both
values within tolerance; the V4 value is lower than the paper's, a larger effect.

**Result: fig6d-tuning-correlation-spacing partially reproduced.** Near against far tuning
correlates less in V4 than V1 (reading a: 0.135 against 0.432, p = 1.2e-06; reading b: 0.214
against 0.515, p = 1.8e-07), but no value is within tolerance of 0.17 +/- 0.02 and 0.36 +/- 0.02.
The random-split variant of the reliability gives the same picture (V1 0.374 against V4 0.132).

In [ ]:
TE, RELE = {}, {}
for area in AREAS:
    ps = pops(area, O)
    RELE[area] = np.concatenate([cw.even_odd_reliability(p) for p in ps])
    c0 = np.concatenate([cw.tuning(p, "crowd", config=0) for p in ps])
    c1 = np.concatenate([cw.tuning(p, "crowd", config=1) for p in ps])
    TE[area] = cw.rowcorr(c0, c1)
    print(f"{area}: {len(RELE[area])} neurons in the orientation experiment; target-alone presentations per orientation "
          f"{min(r.shape[1] for p in ps for r in cw.responses(p, 'alone'))} to {max(r.shape[1] for p in ps for r in cw.responses(p, 'alone'))}, "
          f"crowded per orientation and configuration {min(r.shape[1] for p in ps for c in (0, 1) for r in cw.responses(p, 'crowd', config=c))} "
          f"to {max(r.shape[1] for p in ps for c in (0, 1) for r in cw.responses(p, 'crowd', config=c))}")
for crit in (0.5, 0.7, 0.9):
    out = {area: TE[area][(RELE[area] > crit) & np.isfinite(TE[area])] for area in AREAS}
    print(f"criterion {crit}: V1 {out['V1'].mean():.3f} +/- {out['V1'].std(ddof=1) / np.sqrt(len(out['V1'])):.3f} (n {len(out['V1'])}), "
          f"V4 {out['V4'].mean():.3f} +/- {out['V4'].std(ddof=1) / np.sqrt(len(out['V4'])):.3f} (n {len(out['V4'])}); "
          f"one-sided p = {ttest_1s_greater(out['V1'], out['V4']).pvalue:.2g} (Welch {ttest_1s_greater(out['V1'], out['V4'], False).pvalue:.2g})")
print("Expected: V4 0.24 +/- 0.06, V1 0.65 +/- 0.06, p < 0.001")
oA = [p for p in O if p["area"] == "V4"]
print("Autocorrelation of the population residual, orientation experiment (median lag 10): V1",
      round(float(np.median([lagcorr(resid_series(p), 10) for p in O if p['area'] == 'V1'])), 3),
      "V4", round(float(np.median([lagcorr(resid_series(p), 10) for p in oA])), 3))

**Result: reproduced.** In the orientation experiment, at criterion 0.5, the tuning with the two
distractor configurations correlates 0.644 +/- 0.061 in V1 (50 neurons) and 0.214 +/- 0.047 in
V4 (90 neurons), within 0.65 +/- 0.06 and 0.24 +/- 0.06; V1 exceeds V4 at p = 6.7e-08. The
same holds at criteria 0.7 and 0.9 (p = 2.1e-05 and 0.006).

## figs6-common-neuron-pool

> The distinct V4 decoding axes for different displays do not arise from relying on distinct
> subsets of neurons; most target-responsive neurons remain driven with distractors, and the most
> informative neuron for targets alone remains informative with distractors.

Driven: mean count above the blank mean plus 1 SD of the blank counts (Figure S6 legend), to
targets alone and, for those, to targets with distractors at each spacing. Informativeness: |d'|
between the two extreme targets, normalized by the population's summed |d'| in that condition.

In [ ]:
rows = []
for p in S:
    bl = p["counts"][:, cw.stim_mask(p, "blank")]
    thr = bl.mean(1) + bl.std(1, ddof=1)
    drv = p["counts"][:, cw.stim_mask(p, "alone")].mean(1) > thr
    r = dict(area=p["area"], session=p["session"], orange=p["orange"], n=len(thr), driven=drv.mean())
    for s in SP:
        dc = p["counts"][:, cw.stim_mask(p, "crowd", s)].mean(1) > thr
        r[f"still_{SPL[s]}"] = dc[drv].mean() if drv.any() else np.nan
    first, second = cw.extreme_order(p)
    dps = {}
    for cond, s in (("alone", None), ("near", 1.32), ("far", 2.56)):
        rr = cw.responses(p, "alone" if s is None else "crowd", s)
        d = np.abs(np.array([cw.dprime(rr[first][i], rr[second][i]) for i in range(len(thr))]))
        d[~np.isfinite(d)] = 0
        dps[cond] = d / d.sum()
    top = int(np.argmax(dps["alone"]))
    r.update(top_alone=dps["alone"][top], top_near=dps["near"][top], top_far=dps["far"][top])
    rows.append(r)
s6 = pd.DataFrame(rows)
print(s6.groupby("area")[["driven"] + [f"still_{SPL[s]}" for s in SP]].agg(["mean", "sem"]).round(3).T.to_string())
print("Populations with no target-driven neuron:", s6.groupby("area")["still_1.3"].apply(lambda x: int(x.isna().sum())).to_dict())
for area in AREAS:
    d = s6[s6.area == area]
    for cond in ("near", "far"):
        r_, p_ = stats.pearsonr(d.top_alone, d[f"top_{cond}"])
        print(f"{area} top neuron, alone {d.top_alone.mean():.3f} vs {cond} {d[f'top_{cond}'].mean():.3f}: Pearson r = {r_:.3f}, "
              f"p = {p_:.2g}, n = {len(d)} populations")

**Result: reproduced.** The driven fractions cover only the 10 of 22 populations per area
(sessions 3 to 7) where any unit responds above the epochs labelled blank; in the other 12 no unit
does (data check). In those 10, the mean fraction of target-driven units still driven with
distractors is 0.944 to 0.960 in V1 and 0.866 to 0.953 in V4, at least 0.8 at every spacing. The
top-unit test covers all 22 populations: the most informative unit for targets alone keeps its
share of the population's d' with distractors, across V4 populations r = 0.672 (p = 0.0006) with
near and 0.803 (p = 6.8e-06) with far distractors (V1: 0.855 and 0.921). The fraction driven by
targets (0.372 and 0.396, over the 22 populations) is in the range of Figure S6A, 0.3 to 0.5.


## Population decoding

One L1 logistic decoder per population, pair of target orientations (six pairs) and condition
(targets alone, or with distractors at each spacing), fitted and tested by cross-validation as
the STAR Methods describe. Runs: `main` (10 outer and 10 inner folds), `folds5` (5 and 5, the
variant), `shuffled_all` (each neuron's counts shuffled across trials of the same stimulus before
decoding, reading (a) of fig3c) and `shuffled_train` (training trials shuffled, test trials
intact, reading (b)). Each population's run has its own seeded generator. Timed on two
populations first: about 1 s per decoder for `main`.

In [ ]:
PAIRS = [(0, 1), (1, 2), (2, 3), (0, 2), (1, 3), (0, 3)]
CONDS = [("alone", None)] + [("crowd", s) for s in SP]
MODES = {"main": (10, 10), "folds5": (5, 5), "shuffled_all": (10, 10), "shuffled_train": (10, 10)}


def decode_population(pi, mode):
    p = S[pi]
    outer, inner = MODES[mode]
    rng = np.random.default_rng([7, pi, list(MODES).index(mode)])
    rows = []
    for a, b in PAIRS:
        for kind, s in CONDS:
            X, y = cw.pair_trials(p, kind, s, a, b)
            diag = []
            if mode == "shuffled_all":
                perf = cw.decode(cw.shuffle_within(X, y, rng), y, rng, outer, inner, diag)
            elif mode == "shuffled_train":
                perf = cw.decode_shuffled_train(X, y, rng, outer, inner)
            else:
                perf = cw.decode(X, y, rng, outer, inner, diag)
            rows.append(dict(mode=mode, area=p["area"], animal=p["animal"], session=p["session"], orange=p["orange"],
                             a=a, b=b, sep_deg=6 * (b - a), cond="alone" if s is None else s, perf=perf, n=len(y),
                             n_test=len(y) / outer, Cs=[c for c, _ in diag], nonzero=[k for _, k in diag]))
    return rows


t0 = time.time()
DEC = []
for mode in MODES:
    t1 = time.time()
    res = Parallel(n_jobs=4)(delayed(decode_population)(pi, mode) for pi in range(len(S)))
    DEC += [r for rr in res for r in rr]
    print(f"{mode}: {len(S) * len(PAIRS) * len(CONDS)} decoders in {time.time() - t1:.0f} s")
DEC = pd.DataFrame(DEC)
print(f"decoding total {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
print(DEC.groupby(["mode", "area", "cond"]).perf.agg(["count", "mean", "min", "max"]).round(3).to_string())


def paired_points(mode, area, s, animal_=None):
    d = DEC[(DEC["mode"] == mode) & (DEC.area == area)]
    if animal_ is not None:
        d = d[d.animal == animal_]
    al = d[d.cond == "alone"].set_index(["session", "orange", "a", "b"])
    cr = d[d.cond == s].set_index(["session", "orange", "a", "b"])
    j = al.add_suffix("_alone").join(cr.add_suffix("_crowd"))
    return j.reset_index()

In [ ]:
# Decoder diagnostics (main run): the penalty chosen in each outer fold, and how often the chosen
# penalty leaves no nonzero coefficient (the decoder then predicts one class for every test trial).
dm = DEC[DEC["mode"] == "main"]
rows = []
for (area, cond), d in dm.groupby(["area", "cond"]):
    cs = np.concatenate(d.Cs.values)
    nz = np.concatenate(d.nonzero.values)
    r = dict(area=area, cond=cond, folds=len(cs), frac_all_zero=np.mean(nz == 0), median_nonzero=np.median(nz))
    for c in cw.CS:
        r[f"C={c:.2g}"] = np.mean(np.isclose(cs, c))
    rows.append(r)
print(pd.DataFrame(rows).round(3).to_string())
print("Fraction of folds at the smallest C (largest penalty) and at the largest C, all decoders:",
      round(float(np.mean(np.isclose(np.concatenate(dm.Cs.values), cw.CS[0]))), 3),
      round(float(np.mean(np.isclose(np.concatenate(dm.Cs.values), cw.CS[-1]))), 3))


In 9.9% to 14.5% of the main run's outer folds in V1, and 4.8% (targets alone) to 18.4% (1.3
degrees) in V4, the chosen penalty is the largest of the grid (C = 0.01) and leaves no nonzero
coefficient, so the decoder predicts one class for every held-out trial of the fold. The paper's
rule (the largest penalty that reaches the maximal cross-validated performance) selects such a
decoder whenever no smaller penalty does better on the inner folds. This pulls the weaker
performances to chance, and is one way the decoders here may differ from the paper's.


## fig2de-population-decoding-example

> In one animal at the nearest spacing, V4 population decoding performance with distractors
> tends to be lower than for targets alone when performance is above chance (p < 0.001), while
> in V1 the loss is not significant (p > 0.05).

The animal with 84 cases (6 pairs x 14 populations) is AL; CA has 48. Both are shown.

In [ ]:
for mode in ("main", "folds5"):
    for an in ("AL", "CA"):
        for area in AREAS:
            j = paired_points(mode, area, 1.32, an)
            for restrict in (True, False):
                x = j[j.perf_alone > 0.5] if restrict else j
                d = x.perf_alone - x.perf_crowd
                tt = stats.ttest_rel(x.perf_alone, x.perf_crowd)
                w = stats.wilcoxon(x.perf_alone, x.perf_crowd) if (d != 0).any() else None
                print(f"{mode:6s} {an} {area} {'alone > 0.5' if restrict else 'all cases  '}: n = {len(x):3d} of {len(j)}, "
                      f"alone {x.perf_alone.mean():.3f}, crowded {x.perf_crowd.mean():.3f}, difference {d.mean():+.3f}; "
                      f"paired t p = {tt.pvalue:.2g}, Wilcoxon p = {w.pvalue if w else np.nan:.2g}")
fig, axs = plt.subplots(1, 2, figsize=(8, 3.8))
for ax, area in zip(axs, AREAS):
    j = paired_points("main", area, 1.32, "AL")
    ax.scatter(j.perf_alone, j.perf_crowd, s=10)
    ax.plot([0.3, 1], [0.3, 1], "k:")
    ax.set(xlim=(0.3, 1), ylim=(0.3, 1), xlabel="performance, target alone", ylabel="performance, nearest distractors", title=f"{area}, AL")
plt.tight_layout()
plt.show()
for area in AREAS:
    j = paired_points("main", area, 1.32, "AL")
    print(area, "points outside the axes (below 0.3):", int(((j.perf_alone < 0.3) | (j.perf_crowd < 0.3)).sum()), "of", len(j))

**Result: reproduced, with V1 taken over all cases.** In AL at 1.3 degrees (84 cases), V4
performance with distractors is lower where target-alone performance exceeds 0.5 (74 cases,
0.645 against 0.536, paired t p = 1.5e-15, Wilcoxon p = 2.8e-11). In V1 the difference over all
84 cases is not significant (0.560 against 0.552, p = 0.46 and 0.60), but restricted as V4 is it
is (62 cases, p = 0.018 and 0.025), so the V1 part rests on the unrestricted reading. The
5-fold variant and the other animal give the same pattern.

## Threshold elevation (Figure 2F)

> fig2f-v1-threshold-elevation: In V1, distractors elevate the estimated discrimination threshold
> of the populations by 1.31 +/- 0.10 at the closest spacing and 1.15 +/- 0.08 at the farthest,
> significant at all but the farthest spacing (permutation test, p < 0.05).
>
> fig2f-v4-threshold-elevation: In V4 the estimated threshold elevations are greater than in V1,
> 2.52 +/- 0.23 for the closest distractors and 1.89 +/- 0.19 for the farthest, significant at all
> spacings (permutation test, p < 0.001).

Per animal, area and spacing, k by maximum likelihood over the (target alone, crowded) points of
all pairs and populations; points with target-alone performance at or below 0.5 or at 1 dropped.
Binomial trial count: (a) the held-out trials of one fold, (b) all trials of the pair; since the
held-out count is a tenth of the total for every point, the two scale the likelihood by a constant
and give the same k, so they differ only in the bootstrap and permutation draws. The Figure
2F value is the mean of the two animals' estimates; its SEM is the SD of that mean over 1000
bootstrap samples of each animal's points; the permutation test swaps the two labels at random
within each point, 1000 times, refits both animals and averages; the table gives the median and
the 5th and 95th percentiles of that null. The k grid runs from 0.2 to 20; `k_wide` repeats the
estimate on a grid up to 1000 to show where the bound censors it.

In [ ]:
def elevation(mode, area, s, ntrials, rng, n_boot=1000, n_perm=1000):
    ests, boots, perms, used, wide = [], [], [], [], []
    for an in ("AL", "CA"):
        j = paired_points(mode, area, s, an)
        n = j.n_test_crowd.values if ntrials == "fold" else j.n_crowd.values
        pa, pc = j.perf_alone.values, j.perf_crowd.values
        ests.append(cw.fit_k(pa, pc, n))
        wide.append(cw.fit_k(pa, pc, n, cw.K_GRID_WIDE))
        boots.append(cw.boot_k(pa, pc, n, rng, n_boot) if n_boot else None)
        perms.append(cw.perm_k(pa, pc, n, rng, n_perm) if n_perm else None)
        used.append(f"{int(cw.usable(pa).sum())}/{len(pa)}")
    out = dict(mode=mode, area=area, spacing=SPL[s], ntrials=ntrials, k_AL=ests[0], k_CA=ests[1], k=np.mean(ests),
               points_used=" ".join(used), k_AL_wide=wide[0], k_CA_wide=wide[1], k_wide=np.mean(wide))
    if n_boot:
        out["sem"] = np.std((boots[0] + boots[1]) / 2, ddof=1)
    if n_perm:
        null = (perms[0] + perms[1]) / 2
        out["p_perm"] = (1 + np.sum(null >= out["k"])) / (1 + len(null))
        out["null_median"], out["null_5"], out["null_95"] = np.percentile(null, [50, 5, 95])
    return out


t0 = time.time()
rng = np.random.default_rng(8)
TE_ROWS = [elevation(mode, area, s, nt, rng) for mode in ("main", "folds5") for nt in ("fold", "all")
           for area in AREAS for s in SP]
TH = pd.DataFrame(TE_ROWS)
print(f"threshold fits in {time.time() - t0:.0f} s")
print(TH.to_string(float_format=lambda v: f"{v:.4g}"))
print(f"Estimates at the upper bound of K_GRID ({cw.K_GRID[-1]:g}):", int((TH[["k_AL", "k_CA"]] >= cw.K_GRID[-1]).sum().sum()),
      f"of {2 * len(TH)}; k_wide is the estimate on K_GRID_WIDE (up to {cw.K_GRID_WIDE[-1]:g})")

In [ ]:
# Calibration of the threshold-elevation estimate: with no crowding (true k = 1), crowded and
# target-alone performances drawn as binomial fractions around each case's recorded target-alone
# performance, with the recorded trial count of the pair; k estimated as above, mean of the two
# animals, 200 simulations per area and spacing.
rng = np.random.default_rng(15)
rows = []
for area in AREAS:
    for s in SP:
        ks = []
        for _ in range(200):
            e = []
            for an in ("AL", "CA"):
                j = paired_points("main", area, s, an)
                n = j.n_crowd.values
                pt = np.clip(j.perf_alone.values, 0.5, 1)
                e.append(cw.fit_k(rng.binomial(n, pt) / n, rng.binomial(n, pt) / n, n))
            ks.append(np.mean(e))
        ks = np.array(ks)
        rows.append(dict(area=area, spacing=SPL[s], k_null_mean=ks.mean(), k_null_median=np.median(ks),
                         k_null_5=np.percentile(ks, 5), k_null_95=np.percentile(ks, 95)))
print(pd.DataFrame(rows).round(3).to_string())


With no crowding the estimate is biased upward: its mean is 1.17 to 1.18 in V1 (5th to 95th
percentile 1.07 to 1.29) and 1.09 in V4 (1.02 to 1.16), because points near chance enter the fit
by noise. The permutation nulls lie above 1 for the same reason, with medians of 1.37 to 1.63 in V1
and 1.75 to 4.16 in V4 (main run), so each estimate is tested against a null with the same bias.


In [ ]:
for (mode, nt), d in TH[TH.area == "V1"].groupby(["mode", "ntrials"], sort=False):
    k = d.set_index("spacing")
    vals = abs(k.loc["1.3", "k"] - 1.31) <= 0.10 and abs(k.loc["2.6", "k"] - 1.15) <= 0.08
    sig = [bool(k.loc[x, "p_perm"] < 0.05) for x in ("1.3", "1.6", "2.1", "2.6")]
    print(f"{mode:6s} n={nt:4s}: V1 closest {k.loc['1.3', 'k']:.3f} +/- {k.loc['1.3', 'sem']:.3f}, farthest "
          f"{k.loc['2.6', 'k']:.3f} +/- {k.loc['2.6', 'sem']:.3f}; values within: {vals}; p < 0.05 at 1.3, 1.6, 2.1, 2.6: {sig}")

**Result: not reproduced.** The V1 threshold elevation is 1.685 +/- 0.351 at 1.3 degrees and
1.669 +/- 0.314 at 2.6, outside 1.31 +/- 0.10 and 1.15 +/- 0.08. The permutation test finds only
2.1 degrees significant (p = 0.048 and 0.047 for the two trial counts), not 1.3 or 1.6 (p = 0.18
to 0.42) and not 2.6 (p = 0.059 and 0.062); with 5 folds no spacing is significant. The two trial counts are one reading for the estimate
(same k). The V1 estimates are above the no-crowding calibration (95th percentile 1.26 to 1.29)
but near the medians of their permutation nulls (1.63 at 1.3 degrees, 1.49 at 2.6).

In [ ]:
for (mode, nt), d in TH.groupby(["mode", "ntrials"], sort=False):
    k4 = d[d.area == "V4"].set_index("spacing")
    k1 = d[d.area == "V1"].set_index("spacing")
    vals = abs(k4.loc["1.3", "k"] - 2.52) <= 0.23 and abs(k4.loc["2.6", "k"] - 1.89) <= 0.19
    sig = bool((k4.p_perm < 0.001).all())
    above = bool((k4.k > k1.k).all())
    print(f"{mode:6s} n={nt:4s}: V4 closest {k4.loc['1.3', 'k']:.3f} +/- {k4.loc['1.3', 'sem']:.3f}, farthest "
          f"{k4.loc['2.6', 'k']:.3f} +/- {k4.loc['2.6', 'sem']:.3f}; values within: {vals}; p < 0.001 at all four: {sig} "
          f"(smallest attainable p with 1000 permutations is {1 / 1001:.4f}); V4 above V1 at every spacing: {above}")

**Result: partially reproduced.** V4 elevations exceed V1 at every spacing and the permutation
test gives p = 0.001 (the smallest attainable) at all four spacings, but the values are far
from the paper's: 7.30 +/- 2.47 at 1.3 degrees and 2.61 +/- 0.29 at 2.6, against 2.52 +/- 0.23
and 1.89 +/- 0.19. The CA estimates are large (10.87 and 18.52 at 1.3 and 1.6 degrees); on the grid up to 1000
they are 10.87 and 18.47, so the bound of 20 does not censor the main run (in the 5-fold run CA at
1.6 degrees reaches it, 41.0 on the wider grid). The AL estimates are 3.74 and 1.83 at the closest
and farthest spacings. The V4 estimates are far above their permutation null medians (3.51 at 1.3
degrees, 1.75 at 2.6).

## fig3c-shuffled-threshold-elevation

> When decoders are fit to trial-shuffled responses, distractors still lead to threshold
> elevations similar to those in the original data, so the change in correlations plays a minimal
> role in the information loss.

In [ ]:
rng = np.random.default_rng(9)
rows = []
for mode in ("shuffled_all", "shuffled_train"):
    for nt in ("fold", "all"):
        for area in AREAS:
            for s in SP:
                r = elevation(mode, area, s, nt, rng, n_boot=0, n_perm=0)
                o = TH[(TH["mode"] == "main") & (TH.ntrials == nt) & (TH.area == area) & (TH.spacing == SPL[s])].iloc[0]
                r.update(k_original=o.k, sem_original=o["sem"], within_2sem=abs(r["k"] - o.k) <= 2 * o["sem"])
                rows.append(r)
SH = pd.DataFrame(rows)
print(SH[["mode", "ntrials", "area", "spacing", "k", "k_original", "sem_original", "within_2sem"]].to_string(float_format=lambda v: f"{v:.3f}"))
for (mode, nt), d in SH.groupby(["mode", "ntrials"], sort=False):
    k4, k1 = d[d.area == "V4"].k.values, d[d.area == "V1"].k.values
    print(f"{mode:15s} n={nt:4s}: within twice the SEM at {int(d.within_2sem.sum())} of 8; V4 above V1 at every spacing: {bool((k4 > k1).all())}")

**Result: reproduced.** With shuffled responses the threshold elevations lie within twice the
bootstrap SEM of the original estimates at 8 of 8 area and spacing pairs, and V4 exceeds V1 at
every spacing, under both readings (shuffling all trials or only the training trials) and both
trial counts. The comparison is with this attempt's own original estimates, which differ from
the paper's (fig2f claims). The test is lenient where the SEMs are large (V4 2.0 to 2.5 at 1.3
to 2.1 degrees).

## fig7cd-fixed-decoder-generalization

> Projected onto the decoding axis of the opposite condition, discriminability is largely
> preserved in V1 (d' 0.95 +/- 0.13 same, 0.88 +/- 0.12 cross, p = 0.20) but drops significantly
> in V4 (1.02 +/- 0.13 against 0.67 +/- 0.05, p = 0.002, paired t test).

Per population, two L1 logistic decoders (penalty as above) on the two most different target
orientations, one on targets alone and one with distractors at 1.3 degrees. Ten folds over all
trials of both conditions and all four orientations; in each, both decoders are fitted on the
training trials of the extremes and the held-out trials of all orientations, in both conditions,
are projected onto both axes. d' = (mu2 - mu1) / sqrt(0.5 (s1^2 + s2^2)) per pair of orientations
(signed so that the trained direction is positive). Readings: unit of the test (a) population,
averaged over pairs and both conditions, or (b) population x pair x condition; pairs (all, or
the 18 degree pair alone).

In [ ]:
def fixed_axes(pi):
    p = S[pi]
    rng = np.random.default_rng([10, pi])
    m = cw.stim_mask(p, "alone") | cw.stim_mask(p, "crowd", 1.32)
    idx = np.flatnonzero(m)
    X = p["counts"][:, idx].T.astype(float)
    cond = (p["p"].kind.values[idx] == "crowd").astype(int)
    oi = p["p"].oi.values[idx]
    folds = cw._folds(cond * 10 + oi, 10, rng)
    proj = np.full((len(idx), 2), np.nan)
    for j in range(10):
        tr, te = folds != j, folds == j
        for axis in (0, 1):
            sel = tr & (cond == axis) & np.isin(oi, (0, 3))
            Xa, ya = X[sel], (oi[sel] == 3).astype(int)
            C = cw.choose_C(Xa, ya, 10, rng)
            Xtr, Xte = cw._zscore(Xa, X[te])
            coef, b = cw._fit(Xtr, ya, C)
            proj[te, axis] = Xte @ coef + b
    rows = []
    for a, b_ in PAIRS:
        for c in (0, 1):
            for axis in (0, 1):
                x1 = proj[(cond == c) & (oi == a), axis]
                x2 = proj[(cond == c) & (oi == b_), axis]
                rows.append(dict(area=p["area"], animal=p["animal"], session=p["session"], orange=p["orange"], a=a, b=b_,
                                 sep_deg=6 * (b_ - a), data="alone" if c == 0 else "crowded",
                                 axis="alone" if axis == 0 else "crowded", same=c == axis, dprime=cw.dprime(x2, x1)))
    return rows


t0 = time.time()
F7 = pd.DataFrame([r for rr in Parallel(n_jobs=4)(delayed(fixed_axes)(pi) for pi in range(len(S))) for r in rr])
print(f"fixed-axis decoding in {time.time() - t0:.0f} s; d' undefined in {int(F7.dprime.isna().sum())} of {len(F7)}")
print(F7.groupby(["area", "same", "sep_deg"]).dprime.mean().unstack().round(3).to_string())

In [ ]:
exp7 = {"V1": (0.95, 0.13, 0.88, 0.12), "V4": (1.02, 0.13, 0.67, 0.05)}
for pairs_label, pair_sel in (("all pairs", F7.sep_deg > 0), ("18 deg pair", F7.sep_deg == 18)):
    for unit in ("population", "population x pair x condition"):
        for area in AREAS:
            d = F7[pair_sel & (F7.area == area)]
            if unit == "population":
                g = d.groupby(["session", "orange", "same"]).dprime.mean().unstack()
            else:
                g = d.pivot_table(index=["session", "orange", "a", "b", "data"], columns="same", values="dprime")
            s_, c_ = g[True], g[False]
            t = stats.ttest_rel(s_, c_, nan_policy="omit")
            e = exp7[area]
            ok = [abs(s_.mean() - e[0]) <= e[1], abs(c_.mean() - e[2]) <= e[3]]
            print(f"{pairs_label:11s} {unit:30s} {area}: same {s_.mean():.3f} +/- {s_.sem():.3f}, cross {c_.mean():.3f} +/- "
                  f"{c_.sem():.3f} (n = {len(g)}), paired t p = {t.pvalue:.3g}; values within {ok}")
print("d' per population, 18 degree pair, before averaging (same and cross axis, each the mean of the two conditions):")
g18 = F7[F7.sep_deg == 18].groupby(["area", "session", "orange", "same"]).dprime.mean().unstack()
g18.columns = ["cross", "same"]
print(g18[["same", "cross"]].round(3).to_string())
for an in ("AL", "CA"):
    for area in AREAS:
        d = F7[(F7.area == area) & (F7.animal == an)]
        print(an, area, "same %.3f, cross %.3f" % (d[d.same].dprime.mean(), d[~d.same].dprime.mean()))

**Result: partially reproduced.** The tests hold: in V1 the cross projection does not lower d'
(per population, all pairs: same 0.354 +/- 0.044, cross 0.380 +/- 0.039, p = 0.36), and in V4 it
does (0.411 +/- 0.044 against 0.265 +/- 0.043, p = 0.0004; 18 degree pair 0.710 against 0.467, p =
0.002). The d' values are well below the paper's (0.95, 0.88, 1.02 and 0.67), under every reading. Per
population, before averaging, the 18 degree pair gives same-axis d' of 0.02 to 1.66 in V1 and 0.08
to 1.36 in V4; 17 of 22 V1 populations are below 0.95 and 16 of 22 V4 populations below 1.02.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| data-populations | partially reproduced (counts match the deposit; the stated selection does not regenerate them) |
| fig1b-v4-rf-larger | blocked: data missing |
| fig2a-v1-single-neuron-near | reproduced |
| fig2b-v4-single-neuron-near | reproduced |
| fig2c-farthest-spacing-slopes | reproduced |
| fig2c-v4-worse-all-spacings | reproduced |
| fig2de-population-decoding-example | reproduced (V1 part on all cases) |
| fig2f-v1-threshold-elevation | not reproduced |
| fig2f-v4-threshold-elevation | partially reproduced |
| figs3-pseudo-populations | see `pseudo_populations.ipynb` |
| fig3ab-correlations-reduced | reproduced |
| fig3c-shuffled-threshold-elevation | reproduced |
| fig3d-v1-fano-closest | reproduced |
| fig3e-v4-fano-all-spacings | reproduced |
| fig4a-v1-modulation-indices | partially reproduced |
| fig4b-v4-modulation-indices | reproduced (log-index tests) |
| fig4c-modulation-strength-significant | reproduced |
| fig4c-v4-stronger-than-v1 | reproduced |
| fig4c-v4-maintained-v1-weaker-far | reproduced |
| figs4-rf-alignment | blocked: data missing |
| figs5-additive-vs-multiplicative | reproduced |
| fig5bc-tuning-model-terms | partially reproduced |
| fit-quality | not reproduced |
| fig5-terms-covary | reproduced (rests on the even/odd selection) |
| fig5-negative-multiplicative | partially reproduced |
| fig5de-simulated-discriminability | reproduced (rests on the even/odd selection) |
| fig6c-tuning-correlation-near | partially reproduced |
| fig6d-tuning-correlation-spacing | partially reproduced |
| fig6e-tuning-correlation-configuration | reproduced |
| fig6-well-isolated | blocked: data missing |
| fig7cd-fixed-decoder-generalization | partially reproduced |
| figs6-common-neuron-pool | reproduced |

How the analysis differs from the paper's. The paper's code is not public, so every method is
the Methods' as the claims file reads them, with open details run under each reading. The
decoders' penalty grid, z-scoring and fold assignment, the threshold-elevation grid, and the
handling of zero means are standard choices stated in `crowding.py`. All deposited units are
used, as the stated selection cannot be applied (data-populations). Noise correlations use every
stimulus epoch, as the first epoch of a trial cannot be identified. Even and odd presentations
follow the stored order, which for V4 is not the recording order; single random half splits keep
about as many neurons as the stored-order split in both areas, and the tuning-model claims are repeated
on a selection by reliability averaged over 20 random splits, under which fig5-terms-covary and
fig5de-simulated-discriminability would not hold in V4. The decoders leave no nonzero coefficient
in 5% to 18% of folds, and the threshold-elevation estimate is biased upward when there is no
crowding (1.17 in V1, 1.09 in V4); both are properties of the method as the paper describes it,
not choices made here.

What was found about the data set. (1) In sessions 1, 2 and 8 to 11 the epochs labelled blank
and distractor-only respond as the target epochs do, so the spontaneous rate cannot be measured
and the Methods' selection cannot be checked there. (2) V1 and V4 entries of the same population
hold the same epochs but in different orders; the V1 order is consistent with the recording
order and the V4 order is not, so epochs cannot be matched across areas and even/odd splits of
V4 cannot follow time. 244 V4 units pass the reliability criterion against the paper's 198, while
V1 gives the paper's 146 exactly; single random splits give similar counts (238 and 144 on
average), so the order does not explain the difference. (3) The two epochs of a trial are not marked. (4) The
files hold blank and distractor-only epochs and the spacing of the orientation experiment is not
stored; the description mentions neither. (5) The receptive-field mapping, the per-session
coarse mapping and the waveforms are not deposited.